# Qwen-27B Sequential-Handoff Independent Replication

This confirmatory notebook tests whether the previously frozen L48 → L49 → L55 handoff predicts interventions on new physical configurations and new visual renderings.

[实验设置与运行说明](../../../docs/experiment_notes/11_handoff_development_and_matching/f4436967f887c681.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(total=total, initial=initial, desc=label, unit='cell',
               dynamic_ncols=True, leave=True)
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 23 tracked cells.')


## 1. Lock the falsifiable replication

The layer locations, directions, intervention doses, conditions, random controls, and statistical gates are inherited from the completed parent experiment.


In [ ]:
%%cell_progress 1/23 Mount Drive and locate VisualTorque
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (path for path in PROJECT_CANDIDATES
     if (path / 'toolkit' / 'pyproject.toml').is_file()),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/23 Install and verify the multimodal environment
import importlib
import json
import os
import signal
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)

ANALYSIS_STACK = {
    'numpy': '2.5.1',
    'scipy': '1.18.0',
    'pandas': '3.0.5',
    'matplotlib': '3.11.1',
    'scikit-learn': '1.9.0',
    'tqdm': '4.67.1',
}
ENV_MARKER = Path('/content/visualtorque_inference_analysis_stack_v3.json')
probe_code = '''
import json
import matplotlib
import numpy
import pandas
import scipy
import sklearn
import tqdm
print(json.dumps({
    'numpy': numpy.__version__, 'scipy': scipy.__version__,
    'pandas': pandas.__version__, 'matplotlib': matplotlib.__version__,
    'scikit-learn': sklearn.__version__, 'tqdm': tqdm.__version__,
}, sort_keys=True))
'''

def fresh_process_probe():
    return subprocess.run(
        [sys.executable, '-c', probe_code], text=True,
        capture_output=True, check=False,
    )

try:
    marker = json.loads(ENV_MARKER.read_text(encoding='utf-8'))
except (FileNotFoundError, json.JSONDecodeError):
    marker = None
probe = fresh_process_probe()
stack_ready = (
    marker == ANALYSIS_STACK and probe.returncode == 0
    and json.loads(probe.stdout.strip().splitlines()[-1]) == ANALYSIS_STACK
)
if not stack_ready:
    cell_stage('Repairing the pinned binary analysis stack')
    packages = [f'{name}=={version}' for name, version in ANALYSIS_STACK.items()]
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
        '--force-reinstall', '--only-binary=:all:', *packages,
    ])
    repaired = fresh_process_probe()
    if repaired.returncode != 0:
        raise RuntimeError(
            'The repaired stack still fails in a fresh process.\n'
            + repaired.stderr[-4000:]
        )
    repaired_versions = json.loads(repaired.stdout.strip().splitlines()[-1])
    assert repaired_versions == ANALYSIS_STACK, repaired_versions
    ENV_MARKER.write_text(
        json.dumps(ANALYSIS_STACK, sort_keys=True), encoding='utf-8'
    )
    print(
        '\nPinned binary packages were repaired. Colab will restart once. '
        'After reconnecting, choose Runtime > Run all.', flush=True,
    )
    os.kill(os.getpid(), signal.SIGKILL)

cell_stage('Installing pinned inference dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (module_name == 'visual_torque_toolkit'
            or module_name.startswith('visual_torque_toolkit.')):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import csr_matrix
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = ('_template', '_to_device', '_candidate_plan', '_append_text_prefix')
assert all(hasattr(HuggingFaceMultimodalAdapter, name) for name in required_features)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('numpy / scipy:', np.__version__, scipy.__version__)
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 3/23 Verify GPU and Hugging Face access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


In [ ]:
%%cell_progress 4/23 Lock the independent replication protocol
cell_stage('Freezing new-data generation, prior mechanism, conditions, and decision gates')
import hashlib
import json
from datetime import datetime, timezone

MODEL_KEY = 'qwen27b'
MODEL_ID = 'Qwen/Qwen3.5-27B'
MODEL_REVISION = '373612cb4f9288fcf550b526d86d85febedd9801'
MODEL_DTYPE = 'bfloat16'
DEVICE_MAP = 'auto'
EXPECTED_LAYER_COUNT = 64
UPSTREAM_LAYER = 48
UPSTREAM_METHOD = 'pls4_prototype'
SEMANTIC_LAYER = 49
LABEL_LAYER = 55
CAPTURE_LAYERS = (UPSTREAM_LAYER, SEMANTIC_LAYER, LABEL_LAYER)
SITE_SCOPE = 'final_prompt_token'

SEED = 20261229
EXPANSION_SEED = 20261124
DIRECTION_SEED = 20260901
LABELS = ('A', 'B', 'C')
MAPPING_VARIANTS = ('forward', 'reverse')
DONOR_TASKS = ('weight_more', 'weight_less')
QUESTION_KINDS = ('outcome',)
COHORTS = ('both_correct', 'right_error')
SIDES = ('left', 'right')
BOOTSTRAP_REPLICATES = 10000
ALPHA_LEVEL = 0.05
PATCH_CONDITION_BATCH_SIZE = 1
SMOKE_CENTERED_LOGPROB_TOLERANCE = 0.15
CLAMP_PROJECTION_TOLERANCE = 0.15
NORM_RELATIVE_TOLERANCE = 0.02
BASELINE_REPLAY_MIN_CONCORDANCE = 0.90
BASELINE_REPLAY_MIN_CELL_CONCORDANCE = 0.80

GEOMETRIES_PER_WAVE = 48
MAXIMUM_WAVES = 3
MAXIMUM_GEOMETRIES = GEOMETRIES_PER_WAVE * MAXIMUM_WAVES
MINIMUM_MATCHED_UNITS = 20
TARGET_MATCHED_UNITS = 32
EARLY_STOP_EXACT_GEOMETRIES = 40
MINIMUM_STYLE_FAMILIES = 4
STYLE_BALANCE_TOLERANCES = (0, 1, 2, 3, 4)

PHYSICAL_FEATURES = (
    'units_1', 'distance_1', 'units_2', 'distance_2',
    'total_units', 'unit_difference', 'total_distance',
    'distance_difference', 'smaller_side_torque',
    'larger_side_torque', 'absolute_net_torque',
)
SELF_CONTROL_CONDITIONS = (
    'self_block_l49', 'self_random_block_l49',
    'self_block_l55', 'self_random_block_l55',
)
CORE_INTERVENTION_CONDITIONS = (
    'upstream_repair', 'upstream_repair_block_l49',
    'upstream_repair_random_block_l49',
    'upstream_repair_block_l55',
    'upstream_repair_random_block_l55',
)
CORE_CONDITIONS = SELF_CONTROL_CONDITIONS + CORE_INTERVENTION_CONDITIONS
RIGHT_ONLY_CONDITIONS = (
    'l49_repair', 'l49_repair_block_l55',
    'l49_repair_random_block_l55', 'l55_repair',
    'l55_bypass_after_l49_block',
)
ALL_CONDITIONS = CORE_CONDITIONS + RIGHT_ONLY_CONDITIONS

# These six palettes and two layouts per palette were not used by the parent cohort.
STYLE_SPECS = (
    {'style_id': 'cobalt_low', 'style_family': 'cobalt',
     'background': [239, 245, 255], 'ink': [24, 39, 71],
     'block_fill': [37, 99, 235], 'block_size': 26, 'beam_y': 276},
    {'style_id': 'cobalt_high', 'style_family': 'cobalt',
     'background': [239, 245, 255], 'ink': [24, 39, 71],
     'block_fill': [37, 99, 235], 'block_size': 36, 'beam_y': 300},
    {'style_id': 'coral_low', 'style_family': 'coral',
     'background': [255, 244, 241], 'ink': [78, 37, 31],
     'block_fill': [225, 88, 72], 'block_size': 27, 'beam_y': 280},
    {'style_id': 'coral_high', 'style_family': 'coral',
     'background': [255, 244, 241], 'ink': [78, 37, 31],
     'block_fill': [225, 88, 72], 'block_size': 35, 'beam_y': 297},
    {'style_id': 'forest_low', 'style_family': 'forest',
     'background': [240, 249, 243], 'ink': [28, 59, 40],
     'block_fill': [46, 139, 87], 'block_size': 28, 'beam_y': 274},
    {'style_id': 'forest_high', 'style_family': 'forest',
     'background': [240, 249, 243], 'ink': [28, 59, 40],
     'block_fill': [46, 139, 87], 'block_size': 34, 'beam_y': 302},
    {'style_id': 'lilac_low', 'style_family': 'lilac',
     'background': [249, 244, 255], 'ink': [57, 39, 75],
     'block_fill': [142, 91, 191], 'block_size': 25, 'beam_y': 286},
    {'style_id': 'lilac_high', 'style_family': 'lilac',
     'background': [249, 244, 255], 'ink': [57, 39, 75],
     'block_fill': [142, 91, 191], 'block_size': 37, 'beam_y': 294},
    {'style_id': 'sand_low', 'style_family': 'sand',
     'background': [252, 248, 234], 'ink': [67, 53, 29],
     'block_fill': [194, 143, 45], 'block_size': 29, 'beam_y': 278},
    {'style_id': 'sand_high', 'style_family': 'sand',
     'background': [252, 248, 234], 'ink': [67, 53, 29],
     'block_fill': [194, 143, 45], 'block_size': 33, 'beam_y': 304},
    {'style_id': 'charcoal_low', 'style_family': 'charcoal',
     'background': [245, 245, 245], 'ink': [35, 35, 35],
     'block_fill': [91, 91, 91], 'block_size': 26, 'beam_y': 288},
    {'style_id': 'charcoal_high', 'style_family': 'charcoal',
     'background': [245, 245, 245], 'ink': [35, 35, 35],
     'block_fill': [91, 91, 91], 'block_size': 36, 'beam_y': 292},
)
STYLE_BY_ID = {row['style_id']: row for row in STYLE_SPECS}

PREVIOUS_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_exactphysics_sequential_handoff_blocking_v1'
)
PREVIOUS_RUN_PATH = PREVIOUS_ROOT / 'RUN_COMPLETE.json'
PREVIOUS_PROTOCOL_PATH = PREVIOUS_ROOT / 'frozen_protocol.json'
PREVIOUS_SCENE_PATH = PREVIOUS_ROOT / 'frozen_scene_manifest.jsonl'
PREVIOUS_DECISION_PATH = (
    PREVIOUS_ROOT / 'analysis' / 'sequential_handoff_decision.json'
)
PARENT_ROOT = (
    PROJECT_ROOT / 'results' / 'qwen27b_nettorque_semantic_handoff_mediation_v1'
)
PARENT_ROUTE_PATH = PARENT_ROOT / 'selected_upstream_route.json'
PARENT_DIRECTION_PATH = PARENT_ROOT / 'direction_library.json'
AUDIT_ROOT = (
    PROJECT_ROOT / 'results' / 'qwen27b_error_conditioned_physical_state_audit_v1'
)
AUDIT_OUTCOME_MANIFEST_PATH = AUDIT_ROOT / 'outcome_prompt_manifest.jsonl'
AUDIT_OUTCOME_SHARD_DIR = AUDIT_ROOT / 'outcome_capture_shards'
DEPTH_SUPPORT_PATH = (
    PROJECT_ROOT / 'results' / 'semantic_label_depth_localization_v1'
    / 'depth_support.jsonl'
)
INVERSE_SCENE_DIR = (
    PROJECT_ROOT / 'results' / 'inverse_query_weight_route_specificity_v1'
    / 'rendered_scenes'
)

OUTPUT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_sequential_handoff_independent_replication_v1'
)
PROTOCOL_LOCK_PATH = OUTPUT_ROOT / 'frozen_protocol.json'
GEOMETRY_PATH = OUTPUT_ROOT / 'candidate_geometries.jsonl'
CANDIDATE_PATH = OUTPUT_ROOT / 'rendered_candidates.jsonl'
SCREEN_SHARD_DIR = OUTPUT_ROOT / 'screen_shards'
SCREEN_RECORDS_PATH = OUTPUT_ROOT / 'screen_records.jsonl'
PAIR_TABLE_PATH = OUTPUT_ROOT / 'pair_behavior_table.csv'
EDGE_PATH = OUTPUT_ROOT / 'exact_geometry_edges.csv'
MATCH_MANIFEST_PATH = OUTPUT_ROOT / 'frozen_match_manifest.jsonl'
SCENE_MANIFEST_PATH = OUTPUT_ROOT / 'frozen_scene_manifest.jsonl'
SELECTED_IMAGE_DIR = OUTPUT_ROOT / 'selected_images'
MATCH_AUDIT_PATH = OUTPUT_ROOT / 'exact_matching_audit.json'
BASELINE_PATH = OUTPUT_ROOT / 'baseline_mapping_replay.jsonl'
DIRECTION_LIBRARY_PATH = OUTPUT_ROOT / 'direction_library.json'
SHARD_DIR = OUTPUT_ROOT / 'path_blocking_shards'
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
for directory in (
    OUTPUT_ROOT, SCREEN_SHARD_DIR, SELECTED_IMAGE_DIR,
    SHARD_DIR, ANALYSIS_DIR, FIGURE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

RUNTIME_ROOT = Path('/content/visualtorque_sequential_independent_v1')
RUNTIME_IMAGE_DIR = RUNTIME_ROOT / 'images'
RUNTIME_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def stable_key(value, seed=SEED):
    return hashlib.sha256(f'{seed}:{value!r}'.encode('utf-8')).hexdigest()

def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if torch.is_tensor(value) and value.numel() == 1:
        return value.item()
    raise TypeError(type(value).__name__)

def write_json_atomic(path, value):
    temporary = Path(path).with_suffix(Path(path).suffix + '.tmp')
    temporary.write_text(json.dumps(
        value, indent=2, ensure_ascii=False, default=json_default
    ) + '\n', encoding='utf-8')
    temporary.replace(path)

def write_jsonl_atomic(path, rows):
    temporary = Path(path).with_suffix(Path(path).suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(
                row, sort_keys=True, default=json_default
            ) + '\n')
    temporary.replace(path)

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

required_inputs = (
    PREVIOUS_RUN_PATH, PREVIOUS_PROTOCOL_PATH, PREVIOUS_SCENE_PATH,
    PREVIOUS_DECISION_PATH, PARENT_ROUTE_PATH, PARENT_DIRECTION_PATH,
    AUDIT_OUTCOME_MANIFEST_PATH, AUDIT_OUTCOME_SHARD_DIR,
    DEPTH_SUPPORT_PATH, INVERSE_SCENE_DIR,
)
for required in required_inputs:
    assert Path(required).exists(), f'Missing prerequisite: {required}'

previous_run = json.loads(PREVIOUS_RUN_PATH.read_text(encoding='utf-8'))
assert previous_run['primary_decision'] == (
    'ordered_handoff_mediation_supported_on_frozen_exactphysics_cohort'
)
previous_protocol = json.loads(
    PREVIOUS_PROTOCOL_PATH.read_text(encoding='utf-8')
)['identity']
assert previous_protocol['route'] == {
    'upstream': UPSTREAM_LAYER, 'semantic': SEMANTIC_LAYER,
    'response_code': LABEL_LAYER, 'method': UPSTREAM_METHOD,
}
MATCHED_UPSTREAM_ALPHA = float(previous_protocol['matched_upstream_alpha'])
parent_route = json.loads(PARENT_ROUTE_PATH.read_text(encoding='utf-8'))
parent_direction = json.loads(PARENT_DIRECTION_PATH.read_text(encoding='utf-8'))
PARENT_SEMANTIC_NORM = float(parent_direction['semantic_norm'])
assert parent_route['selected_layer'] == UPSTREAM_LAYER
assert parent_route['selected_method'] == UPSTREAM_METHOD

protocol_identity = {
    'schema_version': 'sequential-handoff-independent-replication-1.0',
    'analysis_status': 'prospective independent configuration-and-render replication',
    'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'previous_run_sha256': sha256_file(PREVIOUS_RUN_PATH),
    'previous_protocol_sha256': sha256_file(PREVIOUS_PROTOCOL_PATH),
    'previous_scene_sha256': sha256_file(PREVIOUS_SCENE_PATH),
    'previous_decision_sha256': sha256_file(PREVIOUS_DECISION_PATH),
    'route': {'upstream': UPSTREAM_LAYER, 'semantic': SEMANTIC_LAYER,
              'response_code': LABEL_LAYER, 'method': UPSTREAM_METHOD},
    'matched_upstream_alpha': MATCHED_UPSTREAM_ALPHA,
    'geometry_sampling': (
        'physics-only deterministic stratified sampling after excluding every '
        'unordered physical geometry in the parent sequential cohort'
    ),
    'geometries_per_wave': GEOMETRIES_PER_WAVE,
    'maximum_waves': MAXIMUM_WAVES,
    'minimum_matched_units': MINIMUM_MATCHED_UNITS,
    'target_matched_units': TARGET_MATCHED_UNITS,
    'early_stop_exact_geometries': EARLY_STOP_EXACT_GEOMETRIES,
    'style_specs': list(STYLE_SPECS),
    'stable_correct': 'both answer mappings predict the exact physical side',
    'stable_error': 'both answer mappings predict the opposite physical side',
    'matching': 'identical unordered physical geometry; one match per geometry',
    'core_conditions': list(CORE_CONDITIONS),
    'right_only_conditions': list(RIGHT_ONLY_CONDITIONS),
    'block_definition': (
        'restore only the intervention-induced displacement on the frozen '
        'one-dimensional mediator coordinate to its untreated value'
    ),
    'negative_control': 'deterministic norm-matched random coordinate',
    'selection_uses_hidden_states': False,
    'selection_uses_current_interventions': False,
    'layers_directions_doses_refit_on_replication': False,
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'seed': SEED,
}
if PROTOCOL_LOCK_PATH.is_file():
    previous = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == protocol_identity, (
        'Existing protocol differs. Use a new versioned output directory.'
    )
else:
    write_json_atomic(PROTOCOL_LOCK_PATH, {
        'identity': protocol_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })
print('Frozen route:', (UPSTREAM_LAYER, SEMANTIC_LAYER, LABEL_LAYER))
print('Frozen L48 dose:', MATCHED_UPSTREAM_ALPHA)
print('New geometry target / maximum:', TARGET_MATCHED_UNITS, MAXIMUM_GEOMETRIES)
print('New render families:', sorted({row['style_family'] for row in STYLE_SPECS}))


In [ ]:
%%cell_progress 5/23 Prepare Dataset v1 on local SSD
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all((path / name).is_file()
                                 for name in REQUIRED_DATASET_FILES)

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [member for member in bundle.infolist() if not member.is_dir()]
            root = DATASET_DIR.resolve()
            for member in tqdm(members, desc='Extracting Dataset v1', unit='file'):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


## 2. Generate independent physical and visual support

Candidate physics are selected without model behavior and after excluding all parent physical configurations.


In [ ]:
%%cell_progress 6/23 Build physics-only candidate geometries
cell_stage('Excluding all parent geometries and stratifying the remaining physics')
from collections import Counter, defaultdict

scene_records = read_jsonl(DATASET_DIR / 'scenes.jsonl')

def is_single_stack_scene(row):
    stacks = row.get('stacks', [])
    counts = Counter(str(stack.get('side')) for stack in stacks)
    return bool(
        str(row.get('metadata', {}).get('family', '')) == 'single_stack'
        and str(row.get('scene_id', '')).startswith('single_')
        and len(stacks) == 2 and counts == {'left': 1, 'right': 1}
    )

single_scenes = [row for row in scene_records if is_single_stack_scene(row)]
assert len(single_scenes) == 900, len(single_scenes)

def scene_features(scene):
    left = next(stack for stack in scene['stacks'] if stack['side'] == 'left')
    right = next(stack for stack in scene['stacks'] if stack['side'] == 'right')
    left_units, right_units = int(left['units']), int(right['units'])
    left_distance, right_distance = int(left['distance']), int(right['distance'])
    left_torque = left_units * left_distance
    right_torque = right_units * right_distance
    net_torque = left_torque - right_torque
    sign = lambda value: int(value > 0) - int(value < 0)
    weight_cue = sign(left_units - right_units)
    distance_cue = sign(left_distance - right_distance)
    torque_sign = sign(net_torque)
    if torque_sign != 0 and weight_cue != torque_sign and distance_cue == torque_sign:
        diagnostic_cell = 'weight_conflict'
    elif torque_sign != 0 and distance_cue != torque_sign and weight_cue == torque_sign:
        diagnostic_cell = 'distance_conflict'
    elif torque_sign != 0 and weight_cue == torque_sign and distance_cue == torque_sign:
        diagnostic_cell = 'both_cues_aligned'
    else:
        diagnostic_cell = 'mixed_nonbalanced'
    geometry = tuple(sorted((
        (left_units, left_distance), (right_units, right_distance)
    )))
    (units_1, distance_1), (units_2, distance_2) = geometry
    torques = sorted((left_torque, right_torque))
    return {
        'left_units': left_units, 'right_units': right_units,
        'left_distance': left_distance, 'right_distance': right_distance,
        'left_torque': left_torque, 'right_torque': right_torque,
        'net_torque': net_torque,
        'outcome': 'left' if net_torque > 0 else ('right' if net_torque < 0 else 'balanced'),
        'weight_cue': weight_cue, 'distance_cue': distance_cue,
        'torque_sign': torque_sign, 'diagnostic_cell': diagnostic_cell,
        'margin_bin': min(abs(net_torque) // 4, 5),
        'pair_geometry': geometry,
        'units_1': units_1, 'distance_1': distance_1,
        'units_2': units_2, 'distance_2': distance_2,
        'total_units': left_units + right_units,
        'unit_difference': abs(left_units - right_units),
        'total_distance': left_distance + right_distance,
        'distance_difference': abs(left_distance - right_distance),
        'smaller_side_torque': torques[0],
        'larger_side_torque': torques[1],
        'absolute_net_torque': abs(net_torque),
    }

def geometry_key(value):
    return json.dumps(value, sort_keys=True, separators=(',', ':'))

prior_scenes = read_jsonl(PREVIOUS_SCENE_PATH)
excluded_geometry_keys = {
    geometry_key(row['pair_geometry']) for row in prior_scenes
}
prior_style_ids = {str(row['style_id']) for row in prior_scenes}
new_style_ids = {str(row['style_id']) for row in STYLE_SPECS}
assert prior_style_ids.isdisjoint(new_style_ids)

by_geometry = defaultdict(list)
features_by_scene = {}
for scene in single_scenes:
    scene_id = str(scene['scene_id'])
    features = scene_features(scene)
    features_by_scene[scene_id] = features
    if features['net_torque'] != 0:
        by_geometry[features['pair_geometry']].append(scene)

eligible = []
for geometry, members in by_geometry.items():
    serialized = geometry_key([list(item) for item in geometry])
    if serialized in excluded_geometry_keys or len(members) < 2:
        continue
    members = sorted(members, key=lambda row: stable_key(('anchor', row['scene_id'])))
    anchor = members[0]
    features = features_by_scene[str(anchor['scene_id'])]
    eligible.append({
        'geometry_id': 'independent_geometry_' + stable_key(geometry)[:20],
        'anchor_scene_id': str(anchor['scene_id']),
        'counterpart_scene_id': str(members[1]['scene_id']),
        'pair_geometry': [list(item) for item in geometry],
        **{key: features[key] for key in (
            'diagnostic_cell', 'margin_bin', *PHYSICAL_FEATURES
        )},
    })

# Physics-only round robin across conflict type and torque-margin bin.
queues = defaultdict(list)
for row in eligible:
    queues[(row['diagnostic_cell'], int(row['margin_bin']))].append(row)
for key in queues:
    queues[key].sort(key=lambda row: stable_key(('geometry-order', row['geometry_id'])))
selected_geometries = []
queue_keys = sorted(queues, key=lambda key: stable_key(('stratum', key)))
while len(selected_geometries) < min(MAXIMUM_GEOMETRIES, len(eligible)):
    changed = False
    for key in queue_keys:
        if queues[key] and len(selected_geometries) < MAXIMUM_GEOMETRIES:
            selected_geometries.append(queues[key].pop(0))
            changed = True
    if not changed:
        break
for rank, row in enumerate(selected_geometries):
    row['physics_only_rank'] = rank
    row['wave'] = 1 + rank // GEOMETRIES_PER_WAVE

assert len(selected_geometries) >= GEOMETRIES_PER_WAVE
assert not ({geometry_key(row['pair_geometry']) for row in selected_geometries}
            & excluded_geometry_keys)
write_jsonl_atomic(GEOMETRY_PATH, selected_geometries)
print('Parent physical geometries excluded:', len(excluded_geometry_keys))
print('New candidate geometries:', len(selected_geometries))
print('By wave:', Counter(row['wave'] for row in selected_geometries))
print('By diagnostic cell:', Counter(row['diagnostic_cell'] for row in selected_geometries))


In [ ]:
%%cell_progress 7/23 Render new visual versions of every candidate
cell_stage('Rendering six unseen palettes with two layouts each')
from dataclasses import replace as dataclass_replace
from PIL import Image, ImageOps
from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)
from visual_torque_toolkit.render import RenderStyle, render_scene
from visual_torque_toolkit.world import Scene, Side, Stack

loader_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR, output_dir=OUTPUT_ROOT / '_unused_loader',
    model_id=MODEL_ID, model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,), question_kinds=('outcome',),
    splits=('probe_train', 'probe_dev', 'probe_test'), batch_size=1,
    generation_every=0, seed=SEED,
)
all_examples = tuple(load_behavior_examples(loader_config))
examples_by_source_scene = defaultdict(list)
for example in all_examples:
    if str(example.scene_id) in features_by_scene and example.question_kind == 'outcome':
        examples_by_source_scene[str(example.scene_id)].append(example)
outcome_example_by_source_scene = {}
for scene_id, examples in examples_by_source_scene.items():
    examples = sorted(examples, key=lambda example: str(example.variant))
    outcome_example_by_source_scene[scene_id] = examples[
        int(stable_key(('prompt-variant', scene_id))[:8], 16) % len(examples)
    ]
assert len(outcome_example_by_source_scene) == len(single_scenes)

scene_by_id = {str(row['scene_id']): row for row in single_scenes}

def scene_object(scene_record, scene_id):
    return Scene(
        scene_id=scene_id,
        stacks=tuple(Stack(
            stack_id=str(stack['stack_id']), side=Side(str(stack['side'])),
            distance=int(stack['distance']), units=int(stack['units']),
        ) for stack in scene_record['stacks']),
        metadata=(('source_dataset_scene_id', str(scene_record['scene_id'])),),
    )

def mirrored_features(features):
    return {
        **features,
        'left_units': features['right_units'],
        'right_units': features['left_units'],
        'left_distance': features['right_distance'],
        'right_distance': features['left_distance'],
        'left_torque': features['right_torque'],
        'right_torque': features['left_torque'],
        'net_torque': -features['net_torque'],
        'outcome': {'left': 'right', 'right': 'left'}[features['outcome']],
        'weight_cue': -features['weight_cue'],
        'distance_cue': -features['distance_cue'],
        'torque_sign': -features['torque_sign'],
    }

candidate_manifest = []
render_jobs = [
    (geometry, style) for geometry in selected_geometries for style in STYLE_SPECS
]
for geometry, style_spec in tqdm(render_jobs, desc='Rendering new candidate pairs', unit='pair'):
    source_id = geometry['anchor_scene_id']
    source_record = scene_by_id[source_id]
    base_features = features_by_scene[source_id]
    mirror = mirrored_features(base_features)
    pair_instance_id = f"{geometry['geometry_id']}__{style_spec['style_id']}"
    anchor_scene_id = pair_instance_id + '__anchor'
    mirror_scene_id = pair_instance_id + '__mirror'
    anchor_path = RUNTIME_IMAGE_DIR / f'{anchor_scene_id}.png'
    mirror_path = RUNTIME_IMAGE_DIR / f'{mirror_scene_id}.png'
    if not anchor_path.is_file():
        style = RenderStyle(
            width=896, height=448, ticks_per_side=6,
            beam_y=int(style_spec['beam_y']),
            block_size=int(style_spec['block_size']), block_gap=3,
            background=tuple(style_spec['background']),
            ink=tuple(style_spec['ink']),
            block_fill=tuple(style_spec['block_fill']),
        )
        render_scene(scene_object(source_record, anchor_scene_id), anchor_path, style)
    if not mirror_path.is_file():
        with Image.open(anchor_path) as image:
            ImageOps.mirror(image.convert('RGB')).save(mirror_path)
    style_vector = [
        float(style_spec['block_size']), float(style_spec['beam_y']),
        *[float(value) for value in style_spec['background']],
        *[float(value) for value in style_spec['ink']],
        *[float(value) for value in style_spec['block_fill']],
    ]
    for orientation, scene_id, image_path, current in (
        ('anchor', anchor_scene_id, anchor_path, base_features),
        ('mirror', mirror_scene_id, mirror_path, mirror),
    ):
        exact = str(current['outcome'])
        opposite = {'left': 'right', 'right': 'left'}[exact]
        candidate_manifest.append({
            'geometry_id': geometry['geometry_id'],
            'pair_instance_id': pair_instance_id,
            'scene_id': scene_id, 'orientation': orientation,
            'source_dataset_scene_id': source_id,
            'wave': int(geometry['wave']),
            'physics_only_rank': int(geometry['physics_only_rank']),
            'style_id': style_spec['style_id'],
            'style_family': style_spec['style_family'],
            'style_variant': style_spec['style_id'].split('_')[-1],
            'style_vector': style_vector,
            'runtime_image_path': str(image_path.resolve()),
            'pair_geometry': geometry['pair_geometry'],
            'exact_semantic': exact, 'shortcut_semantic': opposite,
            **current,
        })

expected_candidates = len(selected_geometries) * len(STYLE_SPECS) * 2
assert len(candidate_manifest) == expected_candidates
assert all(Path(row['runtime_image_path']).is_file() for row in candidate_manifest)
write_jsonl_atomic(CANDIDATE_PATH, candidate_manifest)
print('New rendered orientations:', len(candidate_manifest))
print('New rendered mirror pairs:', len(candidate_manifest) // 2)


In [ ]:
%%cell_progress 8/23 Load pinned Qwen-27B and frozen decoder sites
cell_stage(f'Loading {MODEL_KEY} weights for the matched error-specific handoff test')
import gc
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()

adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID, revision=MODEL_REVISION,
    dtype=MODEL_DTYPE, device_map=DEVICE_MAP,
)
display(adapter.metadata)

def resolve_module_path(root, path):
    value = root
    for component in path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = (
        'model.language_model.layers', 'model.text_model.layers',
        'language_model.model.layers', 'model.model.layers',
        'model.layers',
    )
    for path in candidates:
        try:
            layers = resolve_module_path(model, path)
        except AttributeError:
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return path, layers
    raise RuntimeError('Could not locate decoder layers.')

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT
assert all(0 <= layer < len(DECODER_LAYERS) for layer in CAPTURE_LAYERS)
print('Layer path:', LAYER_PATH)
print('Capture layers:', CAPTURE_LAYERS)


In [ ]:
%%cell_progress 9/23 Define the behavior-only screen
cell_stage('Counterbalancing A/B/C labels before any hidden-state access')
import math

RULE_TEXT = (
    'Each square is one unit of weight. Distance is measured in tick '
    'intervals from the center pivot. Torque equals weight times distance. '
    'Total torque on a side is the sum over stacks on that side.'
)
QUESTION_TEXT = 'Which way will the balance scale tip?'
SEMANTIC_TEXT = {'left': 'left side', 'balanced': 'equal', 'right': 'right side'}

def mapping_by_label(semantics):
    assert set(semantics) == {'left', 'balanced', 'right'}
    return tuple({
        'position': index, 'label': LABELS[index], 'semantic': semantic,
    } for index, semantic in enumerate(semantics))

def screen_target_mappings(row):
    exact, wrong = row['exact_semantic'], row['shortcut_semantic']
    balanced_label = LABELS[int(stable_key(
        ('balanced-label', row['pair_instance_id']), seed=EXPANSION_SEED
    )[:8], 16) % 3]
    side_labels = [label for label in LABELS if label != balanced_label]
    result = {}
    for variant, exact_label, wrong_label in (
        ('forward', side_labels[0], side_labels[1]),
        ('reverse', side_labels[1], side_labels[0]),
    ):
        by_label = {
            exact_label: exact, wrong_label: wrong, balanced_label: 'balanced'
        }
        result[variant] = mapping_by_label(tuple(by_label[label] for label in LABELS))
    return result

def semantics_by_label(mapping):
    return tuple(next(item['semantic'] for item in mapping
                      if item['label'] == label) for label in LABELS)

def screen_prompt(mapping):
    options = '\n'.join(
        f"{item['label']}. {SEMANTIC_TEXT[item['semantic']]}" for item in mapping
    )
    return '\n\n'.join([
        'Answer this question using only the supplied balance-scale scene.',
        RULE_TEXT, 'Question: ' + QUESTION_TEXT, 'Options:\n' + options,
        'Do not explain your reasoning. Reply with exactly one option letter '
        'and nothing else. Valid replies are: A, B, C.',
    ])

def build_screen_examples(row):
    mappings = screen_target_mappings(row)
    source_example = outcome_example_by_source_scene[row['source_dataset_scene_id']]
    base = dataclass_replace(
        source_example, scene_id=row['scene_id'],
        image_path=Path(row['runtime_image_path']).resolve(),
        outcome=row['outcome'], net_torque=int(row['net_torque']),
    )
    examples = {}
    for variant, mapping in mappings.items():
        semantics = semantics_by_label(mapping)
        examples[variant] = dataclass_replace(
            base,
            example_id=f"independent_screen:{row['scene_id']}:{variant}",
            question_id=f"independent_screen_{row['scene_id']}_{variant}",
            variant=f'independent_screen_{variant}', prompt=screen_prompt(mapping),
            labels=LABELS, option_text=tuple(SEMANTIC_TEXT[x] for x in semantics),
            option_semantics=semantics,
            gold_label=LABELS[semantics.index(row['exact_semantic'])],
            gold_semantic=row['exact_semantic'],
        )
    return examples, mappings

def prepare_examples(examples):
    examples = tuple(examples)
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError('A/B/C labels must be single-token candidates.')
    for example in examples[1:]:
        assert adapter._candidate_plan(example) == plan
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for example in examples], add_generation_prompt=True,
        )), fixed_prefix,
    )
    return {'examples': examples, 'inputs': inputs, 'token_by_label': token_by_label}

print('Behavior-only prompt and two answer mappings are frozen.')


In [ ]:
%%cell_progress 10/23 Define resumable behavior screening
cell_stage('Defining candidate-restricted scoring without hidden-state hooks')

def score_records(prepared, logits):
    logprobs = torch.log_softmax(logits[:, -1, :].float(), dim=-1)
    records = []
    for index, example in enumerate(prepared['examples']):
        values = [
            float(logprobs[index, prepared['token_by_label'][label]].detach().cpu())
            for label in example.labels
        ]
        maximum = max(values)
        weights = [math.exp(value - maximum) for value in values]
        probabilities = [value / sum(weights) for value in weights]
        predicted = int(np.argmax(values))
        records.append({
            'candidate_labels': list(example.labels),
            'candidate_semantics': list(example.option_semantics),
            'candidate_logprobs': values,
            'candidate_probabilities': probabilities,
            'predicted_label': example.labels[predicted],
            'predicted_semantic': example.option_semantics[predicted],
            'gold_label': example.gold_label, 'gold_semantic': example.gold_semantic,
        })
    return records

def forward_score_prepared(prepared):
    with torch.inference_mode():
        output = adapter.model(**prepared['inputs'], use_cache=False)
    return score_records(prepared, output.logits)

def semantic_margin(record, positive_semantic, negative_semantic):
    semantics = list(record['candidate_semantics'])
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return float(values[semantics.index(positive_semantic)]
                 - values[semantics.index(negative_semantic)])

candidate_by_pair = defaultdict(list)
candidate_by_scene = {}
for row in candidate_manifest:
    candidate_by_pair[row['pair_instance_id']].append(row)
    candidate_by_scene[row['scene_id']] = row
assert all(len(rows) == 2 for rows in candidate_by_pair.values())

screen_identity = {
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'candidate_sha256': sha256_file(CANDIDATE_PATH),
    'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'mapping_variants': list(MAPPING_VARIANTS),
}
SCREEN_IDENTITY_SHA256 = 'sha256:' + hashlib.sha256(
    json.dumps(screen_identity, sort_keys=True).encode('utf-8')
).hexdigest()

def screen_wave(wave):
    pair_ids = sorted(
        {row['pair_instance_id'] for row in candidate_manifest if row['wave'] == wave},
        key=lambda value: stable_key(('screen-order', value)),
    )
    progress = tqdm(pair_ids, desc=f'Wave {wave} endpoint screen', unit='pair')
    for pair_id in progress:
        shard_path = SCREEN_SHARD_DIR / (stable_key(('screen', pair_id))[:24] + '.json')
        if shard_path.is_file():
            payload = json.loads(shard_path.read_text(encoding='utf-8'))
            assert payload['screen_identity_sha256'] == SCREEN_IDENTITY_SHA256
            assert payload['pair_instance_id'] == pair_id
            assert len(payload['records']) == 4
            continue
        ordered_rows = sorted(
            candidate_by_pair[pair_id],
            key=lambda row: ('left', 'right').index(row['exact_semantic']),
        )
        batch_examples, metadata = [], []
        for row in ordered_rows:
            examples, _ = build_screen_examples(row)
            for variant in MAPPING_VARIANTS:
                batch_examples.append(examples[variant])
                metadata.append((row, variant))
        scores = forward_score_prepared(prepare_examples(batch_examples))
        records = []
        for score, (row, variant) in zip(scores, metadata):
            score.update({
                'geometry_id': row['geometry_id'],
                'pair_instance_id': pair_id, 'scene_id': row['scene_id'],
                'wave': int(row['wave']), 'style_id': row['style_id'],
                'style_family': row['style_family'],
                'orientation': row['orientation'],
                'mapping_variant': variant,
                'exact_semantic': row['exact_semantic'],
                'opposite_semantic': row['shortcut_semantic'],
                'exact_minus_opposite': semantic_margin(
                    score, row['exact_semantic'], row['shortcut_semantic']
                ),
                'is_exact': bool(score['predicted_semantic'] == row['exact_semantic']),
                'is_opposite': bool(
                    score['predicted_semantic'] == row['shortcut_semantic']
                ),
            })
            records.append(score)
        write_json_atomic(shard_path, {
            'screen_identity_sha256': SCREEN_IDENTITY_SHA256,
            'pair_instance_id': pair_id, 'records': records,
        })
        progress.set_postfix_str(
            f"{ordered_rows[0]['style_id']} | "
            + '/'.join(record['predicted_semantic'] for record in records)
        )
    progress.close()

def available_screen_records(maximum_wave):
    records = []
    for pair_id, rows in candidate_by_pair.items():
        if int(rows[0]['wave']) > maximum_wave:
            continue
        shard_path = SCREEN_SHARD_DIR / (stable_key(('screen', pair_id))[:24] + '.json')
        if not shard_path.is_file():
            continue
        payload = json.loads(shard_path.read_text(encoding='utf-8'))
        assert payload['screen_identity_sha256'] == SCREEN_IDENTITY_SHA256
        records.extend(payload['records'])
    return records

def classify_pairs(records):
    records_by_scene = defaultdict(list)
    for record in records:
        records_by_scene[record['scene_id']].append(record)
    scene_status = {}
    for scene_id, current in records_by_scene.items():
        row = candidate_by_scene[scene_id]
        assert {item['mapping_variant'] for item in current} == set(MAPPING_VARIANTS)
        predictions = [item['predicted_semantic'] for item in sorted(
            current, key=lambda item: MAPPING_VARIANTS.index(item['mapping_variant'])
        )]
        if all(value == row['exact_semantic'] for value in predictions):
            status = 'correct'
        elif all(value == row['shortcut_semantic'] for value in predictions):
            status = 'error'
        else:
            status = 'unstable_or_balanced'
        scene_status[scene_id] = {
            'status': status, 'predictions': predictions,
            'minimum_absolute_margin': min(
                abs(float(item['exact_minus_opposite'])) for item in current
            ),
        }
    rows = []
    complete_pair_ids = [pair_id for pair_id, candidates in candidate_by_pair.items()
                         if all(row['scene_id'] in scene_status for row in candidates)]
    for pair_id in complete_pair_ids:
        candidates = candidate_by_pair[pair_id]
        left = next(row for row in candidates if row['exact_semantic'] == 'left')
        right = next(row for row in candidates if row['exact_semantic'] == 'right')
        left_state, right_state = scene_status[left['scene_id']], scene_status[right['scene_id']]
        pattern = f"left_{left_state['status']}__right_{right_state['status']}"
        rows.append({
            'geometry_id': left['geometry_id'], 'pair_instance_id': pair_id,
            'wave': int(left['wave']), 'style_id': left['style_id'],
            'style_family': left['style_family'],
            'style_variant': left['style_variant'],
            'style_vector': left['style_vector'],
            'diagnostic_cell': left['diagnostic_cell'],
            'margin_bin': int(left['margin_bin']),
            'pattern': pattern, 'left_status': left_state['status'],
            'right_status': right_state['status'],
            'pair_minimum_absolute_margin': min(
                left_state['minimum_absolute_margin'],
                right_state['minimum_absolute_margin'],
            ),
        })
    return pd.DataFrame(rows), scene_status

def preliminary_support(frame):
    patterns = frame.groupby('geometry_id')['pattern'].apply(set)
    exact_geometries = [geometry_id for geometry_id, values in patterns.items()
                        if {'left_correct__right_correct',
                            'left_correct__right_error'} <= values]
    subset = frame[frame['geometry_id'].isin(exact_geometries)]
    family_counts = {
        pattern: subset[subset['pattern'].eq(pattern)]['style_family'].nunique()
        for pattern in ('left_correct__right_correct', 'left_correct__right_error')
    }
    return exact_geometries, family_counts

print('Behavior-only scorer and resumable wave helpers are ready.')


In [ ]:
%%cell_progress 11/23 Run behavior-screen smoke tests
cell_stage('Checking one rendered mirror pair under both answer mappings')
smoke_pair_id = sorted(candidate_by_pair)[0]
smoke_rows = sorted(
    candidate_by_pair[smoke_pair_id],
    key=lambda row: ('left', 'right').index(row['exact_semantic']),
)
smoke_examples, smoke_metadata = [], []
for row in smoke_rows:
    examples, mappings = build_screen_examples(row)
    assert set(examples) == set(MAPPING_VARIANTS)
    for variant in MAPPING_VARIANTS:
        smoke_examples.append(examples[variant])
        smoke_metadata.append((row, variant))
smoke_scores = forward_score_prepared(prepare_examples(smoke_examples))
smoke_output = []
for score, (row, variant) in zip(smoke_scores, smoke_metadata):
    smoke_output.append({
        'physical_side': row['exact_semantic'], 'mapping': variant,
        'predicted_semantic': score['predicted_semantic'],
        'exact_minus_opposite': semantic_margin(
            score, row['exact_semantic'], row['shortcut_semantic']
        ),
    })
assert {row['physical_side'] for row in smoke_output} == {'left', 'right'}
display(pd.DataFrame(smoke_output))
print('Smoke pair:', smoke_pair_id)


In [ ]:
%%cell_progress 12/23 Screen adaptive waves and freeze behavior records
cell_stage('Screening new images; no hidden state or intervention is inspected')

WAVES_RUN = 0
for wave in range(1, MAXIMUM_WAVES + 1):
    screen_wave(wave)
    WAVES_RUN = wave
    current_records = available_screen_records(wave)
    current_pair_frame, current_scene_status = classify_pairs(current_records)
    current_geometries, current_family_counts = preliminary_support(current_pair_frame)
    print(
        f'Wave {wave}: {len(current_geometries)} exact-support geometries; '
        f'style coverage {current_family_counts}'
    )
    enough = bool(
        len(current_geometries) >= EARLY_STOP_EXACT_GEOMETRIES
        and min(current_family_counts.values(), default=0) >= MINIMUM_STYLE_FAMILIES
    )
    if enough:
        break

screen_records = available_screen_records(WAVES_RUN)
pair_frame, scene_status = classify_pairs(screen_records)
write_jsonl_atomic(SCREEN_RECORDS_PATH, screen_records)
pair_frame.drop(columns=['style_vector']).to_csv(PAIR_TABLE_PATH, index=False)
exact_geometry_ids, exact_family_counts = preliminary_support(pair_frame)
print('Waves run:', WAVES_RUN)
print('Screened mirror pairs:', len(pair_frame))
display(pair_frame['pattern'].value_counts().to_frame())
print('Geometries with correct and right-error styles:', len(exact_geometry_ids))
print('Style-family coverage:', exact_family_counts)


In [ ]:
%%cell_progress 13/23 Select exact-physics matches
cell_stage('Matching correct and right-error renderings only within identical physics')

required_patterns = (
    'left_correct__right_correct',
    'left_correct__right_error',
)
edge_rows = []
for geometry_id, subset in pair_frame.groupby('geometry_id'):
    correct = subset[subset['pattern'].eq(required_patterns[0])]
    error = subset[subset['pattern'].eq(required_patterns[1])]
    for _, correct_row in correct.iterrows():
        for _, error_row in error.iterrows():
            correct_vector = np.asarray(correct_row['style_vector'], dtype=float)
            error_vector = np.asarray(error_row['style_vector'], dtype=float)
            scale = np.asarray([12, 32] + [255] * 9, dtype=float)
            style_distance = float(np.sqrt(np.mean(
                ((correct_vector - error_vector) / scale) ** 2
            )))
            edge_rows.append({
                'edge_index': len(edge_rows), 'geometry_id': geometry_id,
                'both_correct_pair_instance_id': correct_row['pair_instance_id'],
                'right_error_pair_instance_id': error_row['pair_instance_id'],
                'both_correct_style_family': correct_row['style_family'],
                'right_error_style_family': error_row['style_family'],
                'both_correct_style_id': correct_row['style_id'],
                'right_error_style_id': error_row['style_id'],
                'style_distance': style_distance,
                'tie_break': int(stable_key((
                    'edge', correct_row['pair_instance_id'],
                    error_row['pair_instance_id'],
                ))[:8], 16) / 2**32,
            })

EDGE_COLUMNS = (
    'edge_index', 'geometry_id',
    'both_correct_pair_instance_id', 'right_error_pair_instance_id',
    'both_correct_style_family', 'right_error_style_family',
    'both_correct_style_id', 'right_error_style_id',
    'style_distance', 'tie_break',
)
edge_frame = pd.DataFrame(edge_rows, columns=EDGE_COLUMNS)
edge_frame.to_csv(EDGE_PATH, index=False)

def solve_exact_matching(family_tolerance):
    if edge_frame.empty:
        return edge_frame.copy(), {
            'family_tolerance': family_tolerance, 'matched_units': 0,
            'maximum_family_count_difference': None,
            'mean_style_distance': None,
        }
    edge_count = len(edge_frame)
    matrix_rows, lower, upper = [], [], []
    for geometry_id in sorted(edge_frame['geometry_id'].unique()):
        row = np.zeros(edge_count)
        row[edge_frame.index[edge_frame['geometry_id'].eq(geometry_id)]] = 1.0
        matrix_rows.append(row); lower.append(-np.inf); upper.append(1.0)
    for column in ('both_correct_pair_instance_id', 'right_error_pair_instance_id'):
        for pair_id in sorted(edge_frame[column].unique()):
            row = np.zeros(edge_count)
            row[edge_frame.index[edge_frame[column].eq(pair_id)]] = 1.0
            matrix_rows.append(row); lower.append(-np.inf); upper.append(1.0)
    total = np.ones(edge_count)
    matrix_rows.append(total); lower.append(-np.inf); upper.append(TARGET_MATCHED_UNITS)
    style_families = sorted({row['style_family'] for row in STYLE_SPECS})
    for family in style_families:
        difference = np.asarray([
            int(edge['both_correct_style_family'] == family)
            - int(edge['right_error_style_family'] == family)
            for _, edge in edge_frame.iterrows()
        ], dtype=float)
        matrix_rows.append(difference)
        lower.append(-float(family_tolerance)); upper.append(float(family_tolerance))
    constraints = LinearConstraint(
        csr_matrix(np.vstack(matrix_rows)), np.asarray(lower), np.asarray(upper)
    )
    objective = np.asarray([
        -1000.0 + float(row['style_distance']) + 1e-4 * float(row['tie_break'])
        for _, row in edge_frame.iterrows()
    ])
    result = milp(
        c=objective, integrality=np.ones(edge_count),
        bounds=Bounds(np.zeros(edge_count), np.ones(edge_count)),
        constraints=constraints,
        options={'time_limit': 120.0, 'mip_rel_gap': 0.0},
    )
    assert result.success, (family_tolerance, result.message)
    selected = edge_frame[np.asarray(result.x) > 0.5].copy()
    differences = [abs(
        int((selected['both_correct_style_family'] == family).sum())
        - int((selected['right_error_style_family'] == family).sum())
    ) for family in style_families]
    return selected, {
        'family_tolerance': int(family_tolerance),
        'matched_units': int(len(selected)),
        'maximum_family_count_difference': max(differences, default=0),
        'mean_style_distance': float(selected['style_distance'].mean())
        if len(selected) else None,
    }

solutions = [solve_exact_matching(value) for value in STYLE_BALANCE_TOLERANCES]
search_frame = pd.DataFrame([summary for _, summary in solutions])
display(search_frame)
solutions.sort(key=lambda item: (
    -item[1]['matched_units'], item[1]['family_tolerance'],
    item[1]['mean_style_distance']
    if item[1]['mean_style_distance'] is not None else np.inf,
))
selected_edges, selected_summary = solutions[0]
selected_edges = selected_edges.sort_values(
    ['geometry_id', 'style_distance']
).reset_index(drop=True)
if len(selected_edges) == 0:
    raise RuntimeError(
        'No new exact-physics behavior matches were found after all frozen waves. '
        'Do not relax the mechanism after seeing this result; create a versioned '
        'render-only expansion instead.'
    )
print('Selected independent matches:', selected_summary)


In [ ]:
%%cell_progress 14/23 Freeze the independent cohort and direction support
cell_stage('Persisting selected images before any causal intervention')
import shutil

candidate_by_pair = defaultdict(list)
candidate_by_scene = {}
for row in candidate_manifest:
    candidate_by_pair[row['pair_instance_id']].append(row)
    candidate_by_scene[row['scene_id']] = row

match_manifest = []
scene_manifest = []
for index, edge in selected_edges.iterrows():
    match_id = f'independent_exact_{index:03d}'
    match_manifest.append({
        'match_id': match_id, 'geometry_id': edge['geometry_id'],
        'both_correct_pair_instance_id': edge['both_correct_pair_instance_id'],
        'right_error_pair_instance_id': edge['right_error_pair_instance_id'],
        'both_correct_style_family': edge['both_correct_style_family'],
        'right_error_style_family': edge['right_error_style_family'],
        'both_correct_style_id': edge['both_correct_style_id'],
        'right_error_style_id': edge['right_error_style_id'],
        'style_distance': float(edge['style_distance']),
    })
    for cohort, pair_id in (
        ('both_correct', edge['both_correct_pair_instance_id']),
        ('right_error', edge['right_error_pair_instance_id']),
    ):
        for candidate in candidate_by_pair[pair_id]:
            source_path = Path(candidate['runtime_image_path'])
            destination = SELECTED_IMAGE_DIR / source_path.name
            if (not destination.is_file()
                    or destination.stat().st_size != source_path.stat().st_size):
                shutil.copy2(source_path, destination)
            assert sha256_file(source_path) == sha256_file(destination)
            status = scene_status[candidate['scene_id']]['status']
            expected = (
                'error' if cohort == 'right_error'
                and candidate['exact_semantic'] == 'right' else 'correct'
            )
            assert status == expected, (candidate['scene_id'], status, expected)
            scene_manifest.append({
                **candidate, 'match_id': match_id, 'pair_id': pair_id,
                'cohort': cohort, 'side': candidate['exact_semantic'],
                'baseline_status': status,
                'image_path': str(destination.resolve()),
                'mapping_predictions': scene_status[candidate['scene_id']]['predictions'],
                'minimum_absolute_margin': scene_status[candidate['scene_id']][
                    'minimum_absolute_margin'
                ],
            })

MATCHED_UNITS = len(match_manifest)
write_jsonl_atomic(MATCH_MANIFEST_PATH, match_manifest)
write_jsonl_atomic(SCENE_MANIFEST_PATH, scene_manifest)
assert len(scene_manifest) == 4 * MATCHED_UNITS
assert Counter((row['cohort'], row['side']) for row in scene_manifest) == {
    (cohort, side): MATCHED_UNITS for cohort in COHORTS for side in SIDES
}

replication_geometry_keys = {geometry_key(row['pair_geometry']) for row in scene_manifest}
assert replication_geometry_keys.isdisjoint(excluded_geometry_keys)
assert {row['style_id'] for row in scene_manifest}.isdisjoint(prior_style_ids)
exact_violations = []
for match_id, rows in pd.DataFrame(scene_manifest).groupby('match_id'):
    records = rows.to_dict('records')
    if len({geometry_key(row['pair_geometry']) for row in records}) != 1:
        exact_violations.append((match_id, 'pair_geometry'))
    for feature in PHYSICAL_FEATURES:
        if len({float(row[feature]) for row in records}) != 1:
            exact_violations.append((match_id, feature))
assert not exact_violations, exact_violations[:10]

STYLE_DIVERSITY_GATE = bool(min(
    pd.DataFrame(match_manifest)['both_correct_style_family'].nunique(),
    pd.DataFrame(match_manifest)['right_error_style_family'].nunique(),
) >= MINIMUM_STYLE_FAMILIES)
REPLICATION_POWER_GATE = bool(
    MATCHED_UNITS >= MINIMUM_MATCHED_UNITS and STYLE_DIVERSITY_GATE
)
matching_audit = {
    'matched_units': MATCHED_UNITS,
    'target_matched_units': TARGET_MATCHED_UNITS,
    'minimum_matched_units': MINIMUM_MATCHED_UNITS,
    'replication_power_gate': REPLICATION_POWER_GATE,
    'style_diversity_gate': STYLE_DIVERSITY_GATE,
    'exact_physical_matching': True,
    'maximum_absolute_smd': 0.0,
    'parent_geometry_overlap': 0,
    'parent_style_id_overlap': 0,
    'new_physical_configurations': True,
    'new_render_styles': True,
    'selection_uses_hidden_states': False,
    'selection_uses_current_interventions': False,
    'layers_directions_doses_refit_on_replication': False,
}
write_json_atomic(MATCH_AUDIT_PATH, matching_audit)

outcome_example_by_scene = {}
for row in scene_manifest:
    base = outcome_example_by_source_scene[row['source_dataset_scene_id']]
    outcome_example_by_scene[row['scene_id']] = dataclass_replace(
        base, scene_id=row['scene_id'],
        image_path=Path(row['runtime_image_path']).resolve(),
        outcome=row['exact_semantic'], net_torque=int(row['net_torque']),
    )

direction_support = read_jsonl(DEPTH_SUPPORT_PATH)
DIRECTION_RUNTIME_DIR = Path('/content/visualtorque_semantic_direction_images_v1')
DIRECTION_RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
direction_files = sorted({
    role['image_filename']
    for family in direction_support for role in family['roles'].values()
})
for filename in tqdm(direction_files, desc='Caching frozen direction images', unit='image'):
    source_path = INVERSE_SCENE_DIR / filename
    destination = DIRECTION_RUNTIME_DIR / filename
    assert source_path.is_file(), source_path
    if (not destination.is_file()
            or destination.stat().st_size != source_path.stat().st_size):
        shutil.copy2(source_path, destination)

print('Independent matched quadruplets:', MATCHED_UNITS)
print('Replication power gate:', REPLICATION_POWER_GATE)
display(matching_audit)


## 3. Apply the frozen mechanism without refitting

The old disjoint direction support reconstructs the same L48, L49, and L55 coordinates.


In [ ]:
%%cell_progress 15/23 Define frozen direction and target prompts
cell_stage('Separating physical sides from A/B/C response codes')
import math
from dataclasses import replace as dataclass_replace
from visual_torque_toolkit.behavior.data import BehaviorExample

RULE_TEXT = (
    'Each square is one unit of weight. Distance is measured in tick '
    'intervals from the center pivot. Torque equals weight times distance. '
    'Total torque on a side is the sum over stacks on that side.'
)
QUESTION_TEXT = {
    'outcome': 'Which way will the balance scale tip?',
    'weight_more': 'Which side has more total weight?',
    'weight_less': 'Which side has less total weight?',
}
SEMANTIC_TEXT = {'left': 'left side', 'balanced': 'equal', 'right': 'right side'}

def outcome(value):
    return 'left' if value > 0 else 'right' if value < 0 else 'balanced'

def task_semantic(features, task):
    if task == 'outcome':
        value = int(features['net_torque'])
    elif task == 'weight_more':
        value = int(features['weight_score'])
    elif task == 'weight_less':
        value = -int(features['weight_score'])
    else:
        raise KeyError(task)
    return outcome(value)

def mapping_by_label(semantics):
    assert set(semantics) == {'left', 'balanced', 'right'}
    return tuple({
        'position': index, 'label': LABELS[index], 'semantic': semantic,
    } for index, semantic in enumerate(semantics))

def direction_reference_mapping(row):
    semantics = ['left', 'balanced', 'right']
    offset = int(stable_key(
        f"options:{row['family_id']}", seed=DIRECTION_SEED
    )[:8], 16) % 3
    return mapping_by_label(semantics[offset:] + semantics[:offset])

def target_mappings(row):
    exact, wrong = row['exact_semantic'], row['shortcut_semantic']
    assert {exact, wrong} == {'left', 'right'}
    balanced_label = LABELS[int(stable_key(
        ('balanced-label', row['pair_instance_id']), seed=EXPANSION_SEED
    )[:8], 16) % 3]
    side_labels = [label for label in LABELS if label != balanced_label]
    result = {}
    for variant, exact_label, wrong_label in (
        ('forward', side_labels[0], side_labels[1]),
        ('reverse', side_labels[1], side_labels[0]),
    ):
        by_label = {
            exact_label: exact, wrong_label: wrong, balanced_label: 'balanced'
        }
        result[variant] = mapping_by_label(tuple(by_label[label] for label in LABELS))
    return result

def semantics_by_label(mapping):
    return tuple(next(
        item['semantic'] for item in mapping if item['label'] == label
    ) for label in LABELS)

def label_for(mapping, semantic):
    return next(item['label'] for item in mapping if item['semantic'] == semantic)

def custom_prompt(task, mapping):
    options = '\n'.join(
        f"{item['label']}. {SEMANTIC_TEXT[item['semantic']]}" for item in mapping
    )
    return '\n\n'.join([
        'Answer this question using only the supplied balance-scale scene.',
        RULE_TEXT, 'Question: ' + QUESTION_TEXT[task], 'Options:\n' + options,
        'Do not explain your reasoning. Reply with exactly one option letter '
        'and nothing else. Valid replies are: A, B, C.',
    ])

def make_direction_example(row, role_record, task, mapping, name):
    features = role_record['features']
    semantics = semantics_by_label(mapping)
    gold = task_semantic(features, task)
    return BehaviorExample(
        example_id=f"exactphysics_direction:{row['family_id']}:{name}",
        question_id=f"exactphysics_direction_{row['family_id']}_{name}",
        scene_id=str(role_record['scene_id']), condition=PromptCondition.IMAGE_RULE,
        question_kind=task, variant='exactphysics_direction', split='direction_support',
        problem_type='semantic_direction', outcome=gold,
        net_torque=int(features['net_torque']), prompt=custom_prompt(task, mapping),
        image_path=(DIRECTION_RUNTIME_DIR / role_record['image_filename']).resolve(),
        labels=LABELS, option_text=tuple(SEMANTIC_TEXT[x] for x in semantics),
        option_semantics=semantics, gold_label=LABELS[semantics.index(gold)],
        gold_semantic=gold,
    )

def build_direction_examples(row):
    mapping = direction_reference_mapping(row)
    examples = {}
    for task in DONOR_TASKS:
        for role in ('base', 'aligned_source'):
            key = f'{task}__{role}'
            examples[key] = make_direction_example(
                row, row['roles'][role], task, mapping, key
            )
    return examples, mapping

def build_target_examples(row):
    mappings = target_mappings(row)
    base = outcome_example_by_scene[row['scene_id']]
    examples = {}
    for variant, mapping in mappings.items():
        semantics = semantics_by_label(mapping)
        examples[variant] = dataclass_replace(
            base,
            example_id=f"independent_exactphysics_target:{row['scene_id']}:{variant}",
            question_id=f"independent_exactphysics_target_{row['scene_id']}_{variant}",
            variant=f'independent_exactphysics_{variant}', prompt=custom_prompt('outcome', mapping),
            labels=LABELS, option_text=tuple(SEMANTIC_TEXT[x] for x in semantics),
            option_semantics=semantics,
            gold_label=LABELS[semantics.index(row['exact_semantic'])],
            gold_semantic=row['exact_semantic'],
        )
    return examples, mappings

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        hidden = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(output.last_hidden_state):
        hidden = output.last_hidden_state
    else:
        raise TypeError(f'Unsupported output: {type(output).__name__}')
    assert hidden.ndim == 3
    return hidden

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(type(output).__name__)

def prepare_examples(examples):
    examples = tuple(examples)
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError('A/B/C labels must be single-token candidates.')
    for example in examples[1:]:
        assert adapter._candidate_plan(example) == plan
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for example in examples], add_generation_prompt=True,
        )), fixed_prefix,
    )
    return {'examples': examples, 'inputs': inputs, 'token_by_label': token_by_label}

def final_mask(prepared):
    assert len(prepared['examples']) == 1
    attention = prepared['inputs']['attention_mask'][0].detach().cpu().bool()
    mask = torch.zeros_like(attention)
    mask[int(torch.where(attention)[0][-1])] = True
    return mask

print('Prompt builders ready.')


In [ ]:
%%cell_progress 16/23 Define frozen path interventions and scorers
cell_stage('Defining variable-layer final-token transport')

def score_records(prepared, logits):
    logprobs = torch.log_softmax(logits[:, -1, :].float(), dim=-1)
    records = []
    for index, example in enumerate(prepared['examples']):
        scores = [
            float(logprobs[index, prepared['token_by_label'][label]].detach().cpu())
            for label in example.labels
        ]
        maximum = max(scores)
        weights = [math.exp(value - maximum) for value in scores]
        probabilities = [value / sum(weights) for value in weights]
        predicted = int(np.argmax(scores))
        records.append({
            'candidate_labels': list(example.labels),
            'candidate_semantics': list(example.option_semantics),
            'candidate_logprobs': scores,
            'candidate_probabilities': probabilities,
            'predicted_label': example.labels[predicted],
            'predicted_semantic': example.option_semantics[predicted],
            'gold_label': example.gold_label,
            'gold_semantic': example.gold_semantic,
        })
    return records

def forward_capture(prepared, layers):
    hidden_by_layer = {}
    handles = []
    for layer in layers:
        def hook(_module, _inputs, output, layer=layer):
            hidden_by_layer[layer] = output_hidden_tensor(
                output
            ).detach().cpu().clone()
        handles.append(DECODER_LAYERS[layer].register_forward_hook(hook))
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    return score_records(prepared, output.logits), hidden_by_layer

def capture_examples(examples, layers):
    scores = {}
    hidden = {name: {} for name in examples}
    masks = {}
    for name, example in examples.items():
        prepared = prepare_examples([example])
        masks[name] = final_mask(prepared)
        current_scores, current_hidden = forward_capture(prepared, layers)
        scores[name] = current_scores[0]
        for layer, tensor in current_hidden.items():
            hidden[name][layer] = tensor[0]
        del prepared, current_hidden
    return scores, hidden, masks

def selected_norm(tensor):
    return float(torch.linalg.vector_norm(tensor.float()))

def matched_random(tensor, key):
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(stable_key(key)[:16], 16) % (2**63 - 1))
    noise = torch.randn(tensor.shape, generator=generator, dtype=torch.float32)
    return (noise * (selected_norm(tensor) / selected_norm(noise))).to(tensor.dtype)

def forward_intervention_chunk(prepared, conditions):
    assert len(prepared['examples']) == len(conditions)
    layers = {int(row['layer_index']) for row in conditions}
    assert len(layers) == 1
    layer = next(iter(layers))
    payloads = torch.stack([row['payload_selected'] for row in conditions])
    masks = torch.stack([row['target_mask'] for row in conditions])
    modes = [row['intervention_mode'] for row in conditions]
    counts = []
    def hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = payloads.to(device=hidden.device, dtype=hidden.dtype)
        mask = masks.to(device=hidden.device)
        modified = hidden.clone()
        counts.clear()
        for index, mode in enumerate(modes):
            current = mask[index]
            assert int(current.sum()) == int(payload[index].shape[0]) == 1
            if mode == 'add':
                modified[index, current] = hidden[index, current] + payload[index]
            elif mode == 'replace':
                modified[index, current] = payload[index]
            else:
                raise ValueError(mode)
            counts.append(int(current.sum()))
        return replace_output_hidden(output, modified)
    handle = DECODER_LAYERS[layer].register_forward_hook(hook)
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        handle.remove()
    records = score_records(prepared, output.logits)
    for record, condition, count in zip(records, conditions, counts):
        record.update({
            key: value for key, value in condition.items()
            if key not in ('payload_selected', 'target_mask')
        })
        record['patched_tokens'] = count
    return records

def score_conditions(example, conditions):
    records = []
    grouped = defaultdict(list)
    for condition in conditions:
        grouped[int(condition['layer_index'])].append(condition)
    for layer_conditions in grouped.values():
        for start in range(0, len(layer_conditions), PATCH_CONDITION_BATCH_SIZE):
            actual = layer_conditions[start:start + PATCH_CONDITION_BATCH_SIZE]
            padded = list(actual)
            while len(padded) < PATCH_CONDITION_BATCH_SIZE:
                padded.append(padded[-1])
            prepared = prepare_examples([example] * len(padded))
            records.extend(forward_intervention_chunk(
                prepared, padded
            )[:len(actual)])
            del prepared
    return records

def centered_scores(record):
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return values - values.mean()

def label_score(record, label):
    labels = list(record['candidate_labels'])
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    index = labels.index(label)
    other = np.delete(values, index)
    return float(values[index] - np.logaddexp.reduce(other))

def entropy(record):
    probabilities = np.asarray(record['candidate_probabilities'], dtype=float)
    return float(-np.sum(probabilities * np.log(np.clip(probabilities, 1e-12, 1))))

def cosine_similarity(left, right):
    left = left.float().reshape(-1)
    right = right.float().reshape(-1)
    denominator = torch.linalg.vector_norm(left) * torch.linalg.vector_norm(right)
    assert float(denominator) > 0
    return float(torch.dot(left, right) / denominator)

def rescale_direction(tensor, target_norm):
    norm = selected_norm(tensor)
    assert norm > 0 and target_norm > 0
    return (tensor.float() * (float(target_norm) / norm)).to(tensor.dtype)

def unit_direction(tensor):
    return rescale_direction(tensor, 1.0).float()

def score_l49_and_capture_l55(example, payload_selected, clean_l55, metadata):
    prepared = prepare_examples([example])
    mask = final_mask(prepared)
    captured = {}
    def semantic_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = payload_selected.to(device=hidden.device, dtype=hidden.dtype)
        current = mask.to(hidden.device)
        modified = hidden.clone()
        modified[0, current] = hidden[0, current] + payload
        return replace_output_hidden(output, modified)
    def label_capture_hook(_module, _inputs, output):
        captured['hidden'] = output_hidden_tensor(output).detach().cpu().clone()
    semantic_handle = DECODER_LAYERS[SEMANTIC_LAYER].register_forward_hook(
        semantic_hook
    )
    label_handle = DECODER_LAYERS[LABEL_LAYER].register_forward_hook(
        label_capture_hook
    )
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        semantic_handle.remove()
        label_handle.remove()
    record = score_records(prepared, output.logits)[0]
    downstream = captured['hidden'][0][mask] - clean_l55
    record.update(metadata)
    record['patched_tokens'] = int(mask.sum())
    record['downstream_l55_norm'] = selected_norm(downstream)
    return record, downstream

print('Semantic steering and downstream capture scorers ready.')


def semantic_margin(record, exact_semantic, wrong_semantic):
    semantics = list(record['candidate_semantics'])
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return float(
        values[semantics.index(exact_semantic)]
        - values[semantics.index(wrong_semantic)]
    )

def vector_projection(delta, direction):
    unit = unit_direction(direction).reshape(-1)
    return float(torch.dot(delta.float().reshape(-1), unit))

def forward_add_and_capture(example, layer, payload_selected, capture_layers):
    prepared = prepare_examples([example])
    mask = final_mask(prepared)
    captured = {}
    handles = []
    def intervention_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = payload_selected.to(device=hidden.device, dtype=hidden.dtype)
        current = mask.to(hidden.device)
        assert int(current.sum()) == int(payload.shape[0]) == 1
        modified = hidden.clone()
        modified[0, current] = hidden[0, current] + payload
        return replace_output_hidden(output, modified)
    handles.append(DECODER_LAYERS[layer].register_forward_hook(intervention_hook))
    for capture_layer in capture_layers:
        assert capture_layer > layer, (layer, capture_layer)
        def capture_hook(_module, _inputs, output, capture_layer=capture_layer):
            captured[capture_layer] = output_hidden_tensor(
                output
            ).detach().cpu().clone()[0]
        handles.append(
            DECODER_LAYERS[capture_layer].register_forward_hook(capture_hook)
        )
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    record = score_records(prepared, output.logits)[0]
    return record, {
        capture_layer: tensor[mask] for capture_layer, tensor in captured.items()
    }, int(mask.sum())

print('Downstream state-capture intervention helper ready.')

def clamp_selected_coordinate(selected, untreated, axis):
    untreated = untreated.to(device=selected.device, dtype=selected.dtype)
    axis_float = axis.to(device=selected.device, dtype=torch.float32)
    axis_float = axis_float / torch.linalg.vector_norm(axis_float)
    displacement = selected.float() - untreated.float()
    coefficient = torch.sum(displacement * axis_float, dim=-1, keepdim=True)
    clamped = selected.float() - coefficient * axis_float
    return clamped.to(selected.dtype), float(coefficient.detach().cpu().item())

def forward_path_condition(example, operations, untreated_selected, metadata):
    prepared = prepare_examples([example])
    mask = final_mask(prepared)
    captured, removed = {}, {}
    handles = []

    def make_hook(layer):
        operation = operations.get(layer)
        def hook(_module, _inputs, output):
            hidden = output_hidden_tensor(output)
            current = mask.to(hidden.device)
            selected = hidden[0, current]
            changed = False
            if operation is not None:
                action = operation['action']
                if action == 'add':
                    payload = operation['payload'].to(
                        device=hidden.device, dtype=hidden.dtype
                    )
                    selected = selected + payload
                    changed = True
                elif action == 'clamp_axis':
                    selected, coefficient = clamp_selected_coordinate(
                        selected, operation['untreated'], operation['axis']
                    )
                    removed[layer] = coefficient
                    changed = True
                else:
                    raise ValueError(action)
            captured[layer] = selected.detach().cpu().clone()
            if not changed:
                return None
            modified = hidden.clone()
            modified[0, current] = selected
            return replace_output_hidden(output, modified)
        return hook

    active_layers = sorted(set(operations) | {SEMANTIC_LAYER, LABEL_LAYER})
    for layer in active_layers:
        handles.append(DECODER_LAYERS[layer].register_forward_hook(make_hook(layer)))
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    record = score_records(prepared, output.logits)[0]
    record.update(metadata)
    record['patched_tokens'] = int(mask.sum())
    record['removed_projection_l49'] = float(removed.get(SEMANTIC_LAYER, 0.0))
    record['removed_projection_l55'] = float(removed.get(LABEL_LAYER, 0.0))
    record['l48_payload_norm'] = float(
        selected_norm(operations[UPSTREAM_LAYER]['payload'])
        if UPSTREAM_LAYER in operations else 0.0
    )
    record['l49_axis_norm'] = float(
        selected_norm(operations[SEMANTIC_LAYER]['axis'])
        if (SEMANTIC_LAYER in operations
            and operations[SEMANTIC_LAYER]['action'] == 'clamp_axis') else 0.0
    )
    record['l55_axis_norm'] = float(
        selected_norm(operations[LABEL_LAYER]['axis'])
        if (LABEL_LAYER in operations
            and operations[LABEL_LAYER]['action'] == 'clamp_axis') else 0.0
    )
    return record, captured

def build_path_operations(row, variant, mapping, untreated_selected):
    exact, wrong = row['exact_semantic'], row['shortcut_semantic']
    toward_exact_sign = 1.0 if exact == 'left' else -1.0
    exact_label = label_for(mapping, exact)
    wrong_label = label_for(mapping, wrong)
    upstream_payload = (
        toward_exact_sign * UPSTREAM_DIRECTION * MATCHED_UPSTREAM_ALPHA
    )
    semantic_payload = toward_exact_sign * SEMANTIC_DIRECTION
    code_payload = CODE_DIRECTIONS[(wrong_label, exact_label)]
    random_semantic = matched_random(
        semantic_payload,
        ('sequential-block', row['scene_id'], variant, 'l49'),
    )
    random_code = matched_random(
        code_payload,
        ('sequential-block', row['scene_id'], variant, 'l55'),
    )

    add_upstream = {'action': 'add', 'payload': upstream_payload}
    add_semantic = {'action': 'add', 'payload': semantic_payload}
    add_code = {'action': 'add', 'payload': code_payload}
    block_l49 = {
        'action': 'clamp_axis',
        'untreated': untreated_selected[SEMANTIC_LAYER],
        'axis': semantic_payload,
    }
    random_block_l49 = {
        'action': 'clamp_axis',
        'untreated': untreated_selected[SEMANTIC_LAYER],
        'axis': random_semantic,
    }
    block_l55 = {
        'action': 'clamp_axis',
        'untreated': untreated_selected[LABEL_LAYER],
        'axis': code_payload,
    }
    random_block_l55 = {
        'action': 'clamp_axis',
        'untreated': untreated_selected[LABEL_LAYER],
        'axis': random_code,
    }
    operations = {
        'self_block_l49': {SEMANTIC_LAYER: block_l49},
        'self_random_block_l49': {SEMANTIC_LAYER: random_block_l49},
        'self_block_l55': {LABEL_LAYER: block_l55},
        'self_random_block_l55': {LABEL_LAYER: random_block_l55},
        'upstream_repair': {UPSTREAM_LAYER: add_upstream},
        'upstream_repair_block_l49': {
            UPSTREAM_LAYER: add_upstream, SEMANTIC_LAYER: block_l49,
        },
        'upstream_repair_random_block_l49': {
            UPSTREAM_LAYER: add_upstream, SEMANTIC_LAYER: random_block_l49,
        },
        'upstream_repair_block_l55': {
            UPSTREAM_LAYER: add_upstream, LABEL_LAYER: block_l55,
        },
        'upstream_repair_random_block_l55': {
            UPSTREAM_LAYER: add_upstream, LABEL_LAYER: random_block_l55,
        },
        'l49_repair': {SEMANTIC_LAYER: add_semantic},
        'l49_repair_block_l55': {
            SEMANTIC_LAYER: add_semantic, LABEL_LAYER: block_l55,
        },
        'l49_repair_random_block_l55': {
            SEMANTIC_LAYER: add_semantic, LABEL_LAYER: random_block_l55,
        },
        'l55_repair': {LABEL_LAYER: add_code},
        'l55_bypass_after_l49_block': {
            UPSTREAM_LAYER: add_upstream,
            SEMANTIC_LAYER: block_l49,
            LABEL_LAYER: add_code,
        },
    }
    axes = {
        'semantic': semantic_payload, 'semantic_random': random_semantic,
        'code': code_payload, 'code_random': random_code,
    }
    return operations, axes

print('Sequential add-and-clamp path helper ready.')


In [ ]:
%%cell_progress 17/23 Rebuild frozen L48, L49, and L55 directions
cell_stage('Reconstructing the parent route without new scene outcomes')
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler

left_units, left_raw_norms = [], []
code_units, code_raw_norms = defaultdict(list), []
for row in tqdm(direction_support, desc='L49/L55 directions', unit='family'):
    examples, mapping = build_direction_examples(row)
    scores, hidden, masks = capture_examples(
        examples, (SEMANTIC_LAYER, LABEL_LAYER)
    )
    for task in DONOR_TASKS:
        base_key, source_key = f'{task}__base', f'{task}__aligned_source'
        base_semantic = task_semantic(row['roles']['base']['features'], task)
        source_semantic = task_semantic(
            row['roles']['aligned_source']['features'], task
        )
        semantic_delta = (
            hidden[source_key][SEMANTIC_LAYER][masks[source_key]]
            - hidden[base_key][SEMANTIC_LAYER][masks[base_key]]
        )
        left_delta = semantic_delta if source_semantic == 'left' else -semantic_delta
        left_units.append(unit_direction(left_delta))
        left_raw_norms.append(selected_norm(semantic_delta))
        code_delta = (
            hidden[source_key][LABEL_LAYER][masks[source_key]]
            - hidden[base_key][LABEL_LAYER][masks[base_key]]
        )
        transition = (
            label_for(mapping, base_semantic), label_for(mapping, source_semantic)
        )
        code_units[transition].append(unit_direction(code_delta))
        code_raw_norms.append(selected_norm(code_delta))
    del hidden
    torch.cuda.empty_cache()

SEMANTIC_STEERING_NORM = float(np.median(left_raw_norms))
CODE_STEERING_NORM = float(np.median(code_raw_norms))
SEMANTIC_DIRECTION = rescale_direction(
    torch.stack(left_units).mean(dim=0), SEMANTIC_STEERING_NORM
)
CODE_DIRECTIONS = {
    transition: rescale_direction(
        torch.stack(values).mean(dim=0), CODE_STEERING_NORM
    ) for transition, values in code_units.items()
}

audit_manifest = pd.DataFrame(read_jsonl(AUDIT_OUTCOME_MANIFEST_PATH))
hidden_rows, captured_ids = [], []
for path in tqdm(
    sorted(AUDIT_OUTCOME_SHARD_DIR.glob('outcome_*.pt')),
    desc='Loading prior L48 states', unit='shard',
):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    hidden_rows.append(payload['hidden'][str(UPSTREAM_LAYER)].float())
    captured_ids.extend(str(value) for value in payload['scene_ids'])
assert audit_manifest['scene_id'].astype(str).tolist() == captured_ids
X = torch.cat(hidden_rows, dim=0).numpy()
y = audit_manifest['net_torque'].to_numpy(dtype=float)
fit = audit_manifest['split'].isin(['probe_train', 'probe_dev']).to_numpy()
scaler = StandardScaler().fit(X[fit])
X_fit, y_fit = scaler.transform(X[fit]), y[fit]
pls = PLSRegression(n_components=4, scale=False, max_iter=1000)
pls.fit(X_fit, y_fit.reshape(-1, 1))
scores = pls.transform(X_fit)
latent_delta = scores[y_fit > 0].mean(axis=0) - scores[y_fit < 0].mean(axis=0)
score_centered = scores - scores.mean(axis=0, keepdims=True)
X_centered = X_fit - X_fit.mean(axis=0, keepdims=True)
decoder = np.linalg.lstsq(score_centered, X_centered, rcond=1e-5)[0]
raw_delta = (latent_delta @ decoder) * scaler.scale_
UPSTREAM_DIRECTION = torch.from_numpy(
    (raw_delta / 2).astype('float32')
).reshape(1, -1)

parent_upstream = next(
    row for row in parent_direction['upstream']
    if int(row['layer']) == UPSTREAM_LAYER and row['method'] == UPSTREAM_METHOD
)
assert abs(selected_norm(UPSTREAM_DIRECTION) - float(parent_upstream['norm'])) < 1e-4
assert abs(SEMANTIC_STEERING_NORM - PARENT_SEMANTIC_NORM) < 1e-4
direction_metadata = {
    'schema_version': 'fixed-mediator-matched-directions-1.0',
    'upstream_layer': UPSTREAM_LAYER, 'upstream_method': UPSTREAM_METHOD,
    'upstream_norm': selected_norm(UPSTREAM_DIRECTION),
    'semantic_norm': SEMANTIC_STEERING_NORM,
    'code_norm': CODE_STEERING_NORM,
    'matched_upstream_alpha': MATCHED_UPSTREAM_ALPHA,
    'parent_direction_sha256': sha256_file(PARENT_DIRECTION_PATH),
}
write_json_atomic(DIRECTION_LIBRARY_PATH, direction_metadata)
print(direction_metadata)


In [ ]:
%%cell_progress 18/23 Replay the independently frozen behavior cohorts
cell_stage('Replaying both answer mappings without changing cohort membership')

baseline_rows = []
for row in tqdm(scene_manifest, desc='Matched cohort baselines', unit='scene'):
    examples, mappings = build_target_examples(row)
    scores, _, _ = capture_examples(examples, ())
    for variant in MAPPING_VARIANTS:
        score = scores[variant]
        baseline_rows.append({
            'match_id': row['match_id'], 'pair_id': row['pair_id'],
            'style_family': row['style_family'], 'style_id': row['style_id'],
            'scene_id': row['scene_id'], 'cohort': row['cohort'],
            'side': row['side'], 'mapping_variant': variant,
            'baseline_status': row['baseline_status'],
            'exact_semantic': row['exact_semantic'],
            'shortcut_semantic': row['shortcut_semantic'],
            'predicted_semantic': score['predicted_semantic'],
            'correct': bool(score['predicted_semantic'] == row['exact_semantic']),
            'predicted_opposite': bool(
                score['predicted_semantic'] == row['shortcut_semantic']
            ),
            'clean_margin': semantic_margin(
                score, row['exact_semantic'], row['shortcut_semantic']
            ),
        })
baseline_frame = pd.DataFrame(baseline_rows)
write_jsonl_atomic(BASELINE_PATH, baseline_rows)
baseline_frame['expected_correct'] = baseline_frame['baseline_status'].eq('correct')
baseline_frame['replayed_status'] = np.where(
    baseline_frame['correct'], 'correct',
    np.where(baseline_frame['predicted_opposite'], 'error', 'unstable_or_balanced')
)
baseline_frame['replay_concordant'] = (
    baseline_frame['replayed_status'] == baseline_frame['baseline_status']
)
replay_by_cell = baseline_frame.groupby(
    ['cohort', 'side']
)['replay_concordant'].mean()
BASELINE_REPLAY_CONCORDANCE = float(baseline_frame['replay_concordant'].mean())
MINIMUM_CELL_REPLAY_CONCORDANCE = float(replay_by_cell.min())
BASELINE_REPLAY_GATE = bool(
    BASELINE_REPLAY_CONCORDANCE >= BASELINE_REPLAY_MIN_CONCORDANCE
    and MINIMUM_CELL_REPLAY_CONCORDANCE >= BASELINE_REPLAY_MIN_CELL_CONCORDANCE
)
baseline_summary = baseline_frame.groupby(['cohort', 'side']).agg(
    mapping_runs=('correct', 'size'), accuracy=('correct', 'mean'),
    opposite_rate=('predicted_opposite', 'mean'),
    mean_clean_margin=('clean_margin', 'mean'),
    replay_concordance=('replay_concordant', 'mean'),
).round(4)
display(baseline_summary)
print('Global / minimum-cell replay concordance:',
      BASELINE_REPLAY_CONCORDANCE, MINIMUM_CELL_REPLAY_CONCORDANCE)
print('Baseline replay gate:', BASELINE_REPLAY_GATE)
print('No scene is removed after observing replay behavior.')


In [ ]:
%%cell_progress 19/23 Run parity, clamp, and independence smoke tests
cell_stage('Checking that one-axis clamps remove only the intended displacement')

smoke_row = next(
    row for row in scene_manifest
    if row['cohort'] == 'right_error' and row['side'] == 'right'
)
smoke_examples, smoke_mappings = build_target_examples(smoke_row)
smoke_scores, smoke_hidden, smoke_masks = capture_examples(
    smoke_examples, CAPTURE_LAYERS
)
self_errors = {}
for layer in CAPTURE_LAYERS:
    for variant in MAPPING_VARIANTS:
        record = score_conditions(smoke_examples[variant], [{
            'condition': 'self_replay', 'layer_index': layer,
            'intervention_mode': 'replace',
            'payload_selected': smoke_hidden[variant][layer][smoke_masks[variant]],
            'target_mask': smoke_masks[variant],
        }])[0]
        error = float(np.max(np.abs(
            centered_scores(record) - centered_scores(smoke_scores[variant])
        )))
        assert error < SMOKE_CENTERED_LOGPROB_TOLERANCE
        self_errors[f'{layer}:{variant}'] = error

variant = 'forward'
example, mapping = smoke_examples[variant], smoke_mappings[variant]
untreated = {
    layer: smoke_hidden[variant][layer][smoke_masks[variant]]
    for layer in (SEMANTIC_LAYER, LABEL_LAYER)
}
operations, axes = build_path_operations(smoke_row, variant, mapping, untreated)

self_clamp_49, _ = forward_path_condition(
    example,
    {SEMANTIC_LAYER: {
        'action': 'clamp_axis', 'untreated': untreated[SEMANTIC_LAYER],
        'axis': axes['semantic'],
    }}, untreated, {'condition': 'self_clamp_l49'},
)
self_clamp_55, _ = forward_path_condition(
    example,
    {LABEL_LAYER: {
        'action': 'clamp_axis', 'untreated': untreated[LABEL_LAYER],
        'axis': axes['code'],
    }}, untreated, {'condition': 'self_clamp_l55'},
)
for record in (self_clamp_49, self_clamp_55):
    parity = float(np.max(np.abs(
        centered_scores(record) - centered_scores(smoke_scores[variant])
    )))
    assert parity < SMOKE_CENTERED_LOGPROB_TOLERANCE, parity

blocked49_record, blocked49_hidden = forward_path_condition(
    example, operations['upstream_repair_block_l49'], untreated,
    {'condition': 'upstream_repair_block_l49'},
)
blocked55_record, blocked55_hidden = forward_path_condition(
    example, operations['upstream_repair_block_l55'], untreated,
    {'condition': 'upstream_repair_block_l55'},
)
residual49 = abs(vector_projection(
    blocked49_hidden[SEMANTIC_LAYER] - untreated[SEMANTIC_LAYER], axes['semantic']
))
residual55 = abs(vector_projection(
    blocked55_hidden[LABEL_LAYER] - untreated[LABEL_LAYER], axes['code']
))
assert residual49 < CLAMP_PROJECTION_TOLERANCE, residual49
assert residual55 < CLAMP_PROJECTION_TOLERANCE, residual55
for name in ('semantic', 'code'):
    relative = abs(
        selected_norm(axes[name]) - selected_norm(axes[name + '_random'])
    ) / selected_norm(axes[name])
    assert relative < NORM_RELATIVE_TOLERANCE

assert matching_audit['exact_physical_matching']
assert matching_audit['maximum_absolute_smd'] == 0.0
assert matching_audit['parent_geometry_overlap'] == 0
assert matching_audit['parent_style_id_overlap'] == 0
print('Maximum self-replay error:', max(self_errors.values()))
print('Residual intended projection after L49/L55 clamp:', residual49, residual55)
print('Smoke margins:', {
    'clean': semantic_margin(
        smoke_scores[variant], smoke_row['exact_semantic'],
        smoke_row['shortcut_semantic']
    ),
    'blocked49': semantic_margin(
        blocked49_record, smoke_row['exact_semantic'],
        smoke_row['shortcut_semantic']
    ),
    'blocked55': semantic_margin(
        blocked55_record, smoke_row['exact_semantic'],
        smoke_row['shortcut_semantic']
    ),
})


## 4. Test ordered repair, blocking, and downstream bypass

The primary evidence is the loss of an upstream repair after a physical-axis clamp relative to a norm-matched random-axis clamp, followed by restoration from the downstream L55 coordinate.


In [ ]:
%%cell_progress 20/23 Run the frozen sequential interventions on new scenes
cell_stage('Processing independent exact-physics scenes with resumable per-scene shards')

capture_identity = {
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'scene_manifest_sha256': sha256_file(SCENE_MANIFEST_PATH),
    'direction_sha256': sha256_file(DIRECTION_LIBRARY_PATH),
    'core_conditions': list(CORE_CONDITIONS),
    'right_only_conditions': list(RIGHT_ONLY_CONDITIONS),
}
CAPTURE_SHA256 = 'sha256:' + hashlib.sha256(
    json.dumps(capture_identity, sort_keys=True).encode('utf-8')
).hexdigest()

progress = tqdm(scene_manifest, desc='Sequential handoff blocking', unit='scene')
for row in progress:
    shard_path = SHARD_DIR / f"{row['scene_id']}.json"
    expected_conditions = (
        ALL_CONDITIONS if row['side'] == 'right' else CORE_CONDITIONS
    )
    expected_records = len(MAPPING_VARIANTS) * len(expected_conditions)
    if shard_path.is_file():
        existing = json.loads(shard_path.read_text(encoding='utf-8'))
        assert existing['capture_identity_sha256'] == CAPTURE_SHA256
        assert len(existing['records']) == expected_records
        continue

    examples, mappings = build_target_examples(row)
    records = []
    exact, wrong = row['exact_semantic'], row['shortcut_semantic']
    toward_exact_sign = 1.0 if exact == 'left' else -1.0
    for variant in MAPPING_VARIANTS:
        example, mapping = examples[variant], mappings[variant]
        clean_scores, clean_hidden, clean_masks = capture_examples(
            {variant: example}, CAPTURE_LAYERS
        )
        clean = clean_scores[variant]
        clean_margin = semantic_margin(clean, exact, wrong)
        untreated = {
            layer: clean_hidden[variant][layer][clean_masks[variant]]
            for layer in (SEMANTIC_LAYER, LABEL_LAYER)
        }
        operations, axes = build_path_operations(
            row, variant, mapping, untreated
        )
        for condition in expected_conditions:
            progress.set_postfix_str(
                f"{row['cohort']} | {row['side']} | {condition}"
            )
            record, captured = forward_path_condition(
                example, operations[condition], untreated,
                {'condition': condition},
            )
            margin = semantic_margin(record, exact, wrong)
            l49_shift = toward_exact_sign * vector_projection(
                captured[SEMANTIC_LAYER] - untreated[SEMANTIC_LAYER],
                SEMANTIC_DIRECTION,
            )
            l55_shift = vector_projection(
                captured[LABEL_LAYER] - untreated[LABEL_LAYER],
                axes['code'],
            )
            record.update({
                'match_id': row['match_id'], 'pair_id': row['pair_id'],
                'scene_id': row['scene_id'], 'cohort': row['cohort'],
                'side': row['side'], 'baseline_status': row['baseline_status'],
                'style_family': row['style_family'], 'style_id': row['style_id'],
                'mapping_variant': variant,
                'diagnostic_cell': row['diagnostic_cell'],
                'exact_semantic': exact, 'shortcut_semantic': wrong,
                'clean_margin': clean_margin,
                'margin_shift': margin - clean_margin,
                'l49_toward_exact': l49_shift,
                'l55_toward_exact_code': l55_shift,
                'rescued': bool(record['predicted_semantic'] == exact),
                'induced_opposite': bool(record['predicted_semantic'] == wrong),
            })
            records.append(record)
    write_json_atomic(shard_path, {
        'schema_version': 'sequential-handoff-independent-shard-1.0',
        'capture_identity_sha256': CAPTURE_SHA256,
        'scene_id': row['scene_id'], 'records': records,
    })
    progress.set_postfix_str(
        f"{row['cohort']} | {row['side']} | {row['scene_id']}"
    )
    torch.cuda.empty_cache()
progress.close()
print('Completed scene shards:', len(list(SHARD_DIR.glob('*.json'))))


In [ ]:
%%cell_progress 21/23 Assemble independent mapping-averaged estimands
cell_stage('Reducing answer mappings and deriving repair losses caused by each clamp')

all_records = []
for row in scene_manifest:
    payload = json.loads(
        (SHARD_DIR / f"{row['scene_id']}.json").read_text(encoding='utf-8')
    )
    assert payload['capture_identity_sha256'] == CAPTURE_SHA256
    all_records.extend(payload['records'])
record_frame = pd.DataFrame(all_records)
record_frame.to_csv(ANALYSIS_DIR / 'path_blocking_records.csv', index=False)

estimand_frame = record_frame.groupby(
    ['match_id', 'pair_id', 'scene_id', 'cohort', 'side',
     'baseline_status', 'style_family', 'style_id',
     'diagnostic_cell', 'condition'], as_index=False
).agg(
    margin_shift=('margin_shift', 'mean'),
    l49_toward_exact=('l49_toward_exact', 'mean'),
    l55_toward_exact_code=('l55_toward_exact_code', 'mean'),
    rescued=('rescued', 'mean'),
    induced_opposite=('induced_opposite', 'mean'),
    removed_projection_l49=('removed_projection_l49', 'mean'),
    removed_projection_l55=('removed_projection_l55', 'mean'),
    l48_payload_norm=('l48_payload_norm', 'mean'),
    l49_axis_norm=('l49_axis_norm', 'mean'),
    l55_axis_norm=('l55_axis_norm', 'mean'),
)
estimand_frame.to_csv(
    ANALYSIS_DIR / 'mapping_averaged_path_estimands.csv', index=False
)

index_columns = [
    'match_id', 'pair_id', 'scene_id', 'cohort', 'side',
    'baseline_status', 'style_family', 'style_id', 'diagnostic_cell',
]
margin_wide = estimand_frame.pivot(
    index=index_columns, columns='condition', values='margin_shift'
).reset_index()
rescue_wide = estimand_frame.pivot(
    index=index_columns, columns='condition', values='rescued'
).reset_index()

scene_paths = margin_wide[index_columns].copy()
scene_paths['upstream_total'] = margin_wide['upstream_repair']
scene_paths['upstream_l49_block_loss'] = (
    margin_wide['upstream_repair']
    - margin_wide['upstream_repair_block_l49']
    + margin_wide['self_block_l49']
)
scene_paths['upstream_l49_random_loss'] = (
    margin_wide['upstream_repair']
    - margin_wide['upstream_repair_random_block_l49']
    + margin_wide['self_random_block_l49']
)
scene_paths['upstream_l49_specific_loss'] = (
    scene_paths['upstream_l49_block_loss']
    - scene_paths['upstream_l49_random_loss']
)
scene_paths['upstream_l55_block_loss'] = (
    margin_wide['upstream_repair']
    - margin_wide['upstream_repair_block_l55']
    + margin_wide['self_block_l55']
)
scene_paths['upstream_l55_random_loss'] = (
    margin_wide['upstream_repair']
    - margin_wide['upstream_repair_random_block_l55']
    + margin_wide['self_random_block_l55']
)
scene_paths['upstream_l55_specific_loss'] = (
    scene_paths['upstream_l55_block_loss']
    - scene_paths['upstream_l55_random_loss']
)
scene_paths['l49_total'] = margin_wide.get('l49_repair')
scene_paths['l49_l55_block_loss'] = (
    margin_wide.get('l49_repair')
    - margin_wide.get('l49_repair_block_l55')
    + margin_wide['self_block_l55']
)
scene_paths['l49_l55_random_loss'] = (
    margin_wide.get('l49_repair')
    - margin_wide.get('l49_repair_random_block_l55')
    + margin_wide['self_random_block_l55']
)
scene_paths['l49_l55_specific_loss'] = (
    scene_paths['l49_l55_block_loss'] - scene_paths['l49_l55_random_loss']
)
scene_paths['l55_direct'] = margin_wide.get('l55_repair')
scene_paths['l55_bypass_gain'] = (
    margin_wide.get('l55_bypass_after_l49_block')
    - margin_wide['upstream_repair_block_l49']
)
for condition in ALL_CONDITIONS:
    if condition in rescue_wide.columns:
        scene_paths[f'rescued__{condition}'] = rescue_wide[condition]
scene_paths.to_csv(ANALYSIS_DIR / 'scene_level_path_estimands.csv', index=False)

assert scene_paths['match_id'].nunique() == MATCHED_UNITS
assert scene_paths.groupby(['cohort', 'side']).size().eq(MATCHED_UNITS).all()
display(scene_paths.groupby(['cohort', 'side'])[
    ['upstream_total', 'upstream_l49_block_loss',
     'upstream_l49_specific_loss', 'upstream_l55_block_loss',
     'upstream_l55_specific_loss', 'l49_l55_specific_loss',
     'l55_bypass_gain']
].mean().round(4))


In [ ]:
%%cell_progress 22/23 Test the frozen claim and replication robustness
cell_stage('Bootstrapping matched error, correct, side-control, and interaction contrasts')

def bootstrap_summary(values, key):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    assert len(values) > 0
    rng = np.random.default_rng(
        int(stable_key(('bootstrap', key))[:16], 16) % (2**63 - 1)
    )
    draws = rng.choice(
        values, size=(BOOTSTRAP_REPLICATES, len(values)), replace=True
    ).mean(axis=1)
    return {
        'estimate': float(values.mean()),
        'ci_low': float(np.quantile(draws, ALPHA_LEVEL / 2)),
        'ci_high': float(np.quantile(draws, 1 - ALPHA_LEVEL / 2)),
        'n_matched_units': int(len(values)),
    }

def positive(result):
    return bool(result['ci_low'] > 0)

def matched_path_contrasts(frame, metric):
    wide = frame.pivot(
        index='match_id', columns=['cohort', 'side'], values=metric
    ).sort_index()
    both_gap = wide[('both_correct', 'right')] - wide[('both_correct', 'left')]
    error_gap = wide[('right_error', 'right')] - wide[('right_error', 'left')]
    return {
        'right_error_mean': wide[('right_error', 'right')],
        'right_correct_mean': wide[('both_correct', 'right')],
        'right_error_minus_right_correct': (
            wide[('right_error', 'right')] - wide[('both_correct', 'right')]
        ),
        'left_control_difference': (
            wide[('right_error', 'left')] - wide[('both_correct', 'left')]
        ),
        'difference_in_differences': error_gap - both_gap,
    }

core_metrics = (
    'upstream_total',
    'upstream_l49_block_loss', 'upstream_l49_random_loss',
    'upstream_l49_specific_loss',
    'upstream_l55_block_loss', 'upstream_l55_random_loss',
    'upstream_l55_specific_loss',
)
right_only_metrics = (
    'l49_total', 'l49_l55_block_loss', 'l49_l55_random_loss',
    'l49_l55_specific_loss', 'l55_direct', 'l55_bypass_gain',
)
test_rows = []
for metric in core_metrics:
    contrasts = matched_path_contrasts(scene_paths, metric)
    for contrast, values in contrasts.items():
        test_rows.append({
            'metric': metric, 'contrast': contrast,
            **bootstrap_summary(values, (metric, contrast)),
        })
for metric in right_only_metrics:
    current = scene_paths[scene_paths['side'].eq('right')].pivot(
        index='match_id', columns='cohort', values=metric
    ).sort_index()
    contrasts = {
        'right_error_mean': current['right_error'],
        'right_correct_mean': current['both_correct'],
        'right_error_minus_right_correct': (
            current['right_error'] - current['both_correct']
        ),
    }
    for contrast, values in contrasts.items():
        test_rows.append({
            'metric': metric, 'contrast': contrast,
            **bootstrap_summary(values, (metric, contrast)),
        })
test_frame = pd.DataFrame(test_rows)
test_frame.to_csv(ANALYSIS_DIR / 'sequential_path_tests.csv', index=False)

lookup_frame = test_frame.set_index(['metric', 'contrast'])
def result(metric, contrast='right_error_mean'):
    row = lookup_frame.loc[(metric, contrast)]
    return {
        'estimate': float(row['estimate']),
        'ci_low': float(row['ci_low']),
        'ci_high': float(row['ci_high']),
    }

primary_gates = {
    'upstream_repair_positive': positive(result('upstream_total')),
    'l48_to_l49_block_loss_positive': positive(
        result('upstream_l49_block_loss')
    ),
    'l48_to_l49_direction_specific': positive(
        result('upstream_l49_specific_loss')
    ),
    'l48_to_l55_block_loss_positive': positive(
        result('upstream_l55_block_loss')
    ),
    'l48_to_l55_direction_specific': positive(
        result('upstream_l55_specific_loss')
    ),
    'l49_to_l55_block_loss_positive': positive(
        result('l49_l55_block_loss')
    ),
    'l49_to_l55_direction_specific': positive(
        result('l49_l55_specific_loss')
    ),
    'direct_l55_repair_positive': positive(result('l55_direct')),
    'downstream_bypass_positive': positive(result('l55_bypass_gain')),
}

ordered_edges = all(primary_gates[key] for key in (
    'l48_to_l49_block_loss_positive',
    'l48_to_l49_direction_specific',
    'l48_to_l55_block_loss_positive',
    'l48_to_l55_direction_specific',
    'l49_to_l55_block_loss_positive',
    'l49_to_l55_direction_specific',
))
if (
    ordered_edges
    and primary_gates['upstream_repair_positive']
    and primary_gates['direct_l55_repair_positive']
    and primary_gates['downstream_bypass_positive']
):
    PRELIMINARY_DECISION = 'ordered_l48_l49_l55_answer_mediation_supported'
elif (
    primary_gates['l48_to_l55_direction_specific']
    and primary_gates['l49_to_l55_direction_specific']
):
    PRELIMINARY_DECISION = 'late_l55_mediation_supported_without_full_l49_ordering'
elif (
    primary_gates['l48_to_l49_block_loss_positive']
    or primary_gates['l48_to_l55_block_loss_positive']
) and not (
    primary_gates['l48_to_l49_direction_specific']
    or primary_gates['l48_to_l55_direction_specific']
):
    PRELIMINARY_DECISION = 'blocking_effect_is_not_direction_specific'
else:
    PRELIMINARY_DECISION = 'sequential_one_axis_mediation_not_supported'

display(test_frame[
    test_frame['contrast'].isin([
        'right_error_mean', 'right_error_minus_right_correct',
        'difference_in_differences',
    ])
].round(4))
print('Primary gates:', primary_gates)
print('Preliminary decision:', PRELIMINARY_DECISION)


cell_stage('Checking mappings, unseen styles, and leave-one-match-out replication')

def derive_variant_paths(frame):
    index = [
        'match_id', 'scene_id', 'cohort', 'side',
        'style_family', 'mapping_variant',
    ]
    wide = frame.pivot(
        index=index, columns='condition', values='margin_shift'
    ).reset_index()
    result_frame = wide[index].copy()
    result_frame['upstream_l49_specific_loss'] = (
        (wide['upstream_repair'] - wide['upstream_repair_block_l49']
         + wide['self_block_l49'])
        - (wide['upstream_repair'] - wide['upstream_repair_random_block_l49']
           + wide['self_random_block_l49'])
    )
    result_frame['upstream_l55_specific_loss'] = (
        (wide['upstream_repair'] - wide['upstream_repair_block_l55']
         + wide['self_block_l55'])
        - (wide['upstream_repair'] - wide['upstream_repair_random_block_l55']
           + wide['self_random_block_l55'])
    )
    result_frame['l49_l55_specific_loss'] = (
        (wide.get('l49_repair') - wide.get('l49_repair_block_l55')
         + wide['self_block_l55'])
        - (wide.get('l49_repair') - wide.get('l49_repair_random_block_l55')
           + wide['self_random_block_l55'])
    )
    return result_frame

variant_paths = derive_variant_paths(record_frame)
mapping_rows = []
for variant in MAPPING_VARIANTS:
    current = variant_paths[variant_paths['mapping_variant'].eq(variant)]
    for metric in (
        'upstream_l49_specific_loss', 'upstream_l55_specific_loss',
        'l49_l55_specific_loss',
    ):
        values = current[
            current['cohort'].eq('right_error') & current['side'].eq('right')
        ].set_index('match_id')[metric]
        mapping_rows.append({
            'mapping_variant': variant, 'metric': metric,
            **bootstrap_summary(values, ('mapping', variant, metric)),
        })
mapping_frame = pd.DataFrame(mapping_rows)
mapping_frame.to_csv(ANALYSIS_DIR / 'mapping_robustness.csv', index=False)

match_style = {
    row['match_id']: {
        'both_correct': row['both_correct_style_family'],
        'right_error': row['right_error_style_family'],
    } for row in match_manifest
}
style_families = sorted({
    family for values in match_style.values() for family in values.values()
})
style_rows = []
for held_out in style_families:
    kept = [
        match_id for match_id, values in match_style.items()
        if held_out not in values.values()
    ]
    for metric in (
        'upstream_l49_specific_loss', 'upstream_l55_specific_loss',
        'l49_l55_specific_loss',
    ):
        current = scene_paths[
            scene_paths['match_id'].isin(kept)
            & scene_paths['cohort'].eq('right_error')
            & scene_paths['side'].eq('right')
        ].set_index('match_id')[metric]
        if len(current) == 0:
            continue
        style_rows.append({
            'held_out_style_family': held_out, 'metric': metric,
            **bootstrap_summary(current, ('style', held_out, metric)),
        })
style_frame = pd.DataFrame(style_rows)
style_frame.to_csv(ANALYSIS_DIR / 'leave_one_style_family_out.csv', index=False)

leave_match_rows = []
for held_out in sorted(scene_paths['match_id'].unique()):
    current = scene_paths[
        scene_paths['match_id'].ne(held_out)
        & scene_paths['cohort'].eq('right_error')
        & scene_paths['side'].eq('right')
    ]
    leave_match_rows.append({
        'held_out_match_id': held_out,
        **{
            metric: float(current[metric].mean())
            for metric in (
                'upstream_l49_specific_loss',
                'upstream_l55_specific_loss',
                'l49_l55_specific_loss',
            )
        },
    })
leave_match_frame = pd.DataFrame(leave_match_rows)
leave_match_frame.to_csv(
    ANALYSIS_DIR / 'leave_one_match_out.csv', index=False
)

rescue_rows = []
for condition in ALL_CONDITIONS:
    column = f'rescued__{condition}'
    if column not in scene_paths:
        continue
    for cohort, side in (
        ('right_error', 'right'), ('both_correct', 'right'),
        ('right_error', 'left'),
    ):
        values = scene_paths[
            scene_paths['cohort'].eq(cohort)
            & scene_paths['side'].eq(side)
        ][column]
        values = values[np.isfinite(values)]
        if len(values) == 0:
            continue
        rescue_rows.append({
            'condition': condition, 'cohort': cohort, 'side': side,
            **bootstrap_summary(values, ('rescue', condition, cohort, side)),
        })
rescue_frame = pd.DataFrame(rescue_rows)
rescue_frame.to_csv(ANALYSIS_DIR / 'behavioral_rescue_tests.csv', index=False)

robustness_gates = {
    'both_mappings_l49_specific_positive': bool(
        (mapping_frame[
            mapping_frame['metric'].eq('upstream_l49_specific_loss')
        ]['ci_low'] > 0).all()
    ),
    'both_mappings_l55_specific_positive': bool(
        (mapping_frame[
            mapping_frame['metric'].eq('upstream_l55_specific_loss')
        ]['ci_low'] > 0).all()
    ),
    'both_mappings_l49_to_l55_specific_positive': bool(
        (mapping_frame[
            mapping_frame['metric'].eq('l49_l55_specific_loss')
        ]['ci_low'] > 0).all()
    ),
    'all_style_holdouts_l55_positive': bool(
        len(style_frame) > 0 and (style_frame[
            style_frame['metric'].eq('upstream_l55_specific_loss')
        ]['estimate'] > 0).all()
    ),
    'all_match_deletions_l55_positive': bool(
        (leave_match_frame['upstream_l55_specific_loss'] > 0).all()
    ),
}
if not BASELINE_REPLAY_GATE:
    PRIMARY_DECISION = 'replication_invalid_due_to_baseline_replay_drift'
elif (
    PRELIMINARY_DECISION == 'ordered_l48_l49_l55_answer_mediation_supported'
    and robustness_gates['both_mappings_l49_specific_positive']
    and robustness_gates['both_mappings_l55_specific_positive']
    and robustness_gates['both_mappings_l49_to_l55_specific_positive']
    and robustness_gates['all_match_deletions_l55_positive']
):
    PRIMARY_DECISION = (
        'ordered_handoff_mediation_independently_replicated'
        if REPLICATION_POWER_GATE else
        'ordered_handoff_directionally_replicated_but_underpowered'
    )
else:
    PRIMARY_DECISION = PRELIMINARY_DECISION

display(mapping_frame.round(4))
display(style_frame.round(4))
display(rescue_frame.round(4))
print('Robustness gates:', robustness_gates)
print('Primary decision:', PRIMARY_DECISION)


In [ ]:
%%cell_progress 23/23 Audit independence and save the paper-facing report
cell_stage('Auditing independent physics, renders, frozen settings, and records')

assert len(match_manifest) == MATCHED_UNITS
assert len(scene_manifest) == 4 * MATCHED_UNITS
assert matching_audit['maximum_absolute_smd'] == 0.0
assert record_frame['scene_id'].nunique() == 4 * MATCHED_UNITS
assert set(record_frame['mapping_variant']) == set(MAPPING_VARIANTS)
assert record_frame['patched_tokens'].eq(1).all()
assert record_frame.groupby(['scene_id', 'mapping_variant', 'condition']).size().eq(1).all()

left_conditions = set(record_frame[record_frame['side'].eq('left')]['condition'])
right_conditions = set(record_frame[record_frame['side'].eq('right')]['condition'])
assert left_conditions == set(CORE_CONDITIONS)
assert right_conditions == set(ALL_CONDITIONS)
expected_record_count = (
    2 * MATCHED_UNITS * len(MAPPING_VARIANTS) * len(CORE_CONDITIONS)
    + 2 * MATCHED_UNITS * len(MAPPING_VARIANTS) * len(ALL_CONDITIONS)
)
assert len(record_frame) == expected_record_count

norm_audit_rows = []
for keys, current in record_frame.groupby(['scene_id', 'mapping_variant']):
    by_condition = current.set_index('condition')
    semantic_norms = [
        float(by_condition.loc[name, 'l49_axis_norm'])
        for name in (
            'upstream_repair_block_l49',
            'upstream_repair_random_block_l49',
        )
    ]
    code_norms = [
        float(by_condition.loc[name, 'l55_axis_norm'])
        for name in (
            'upstream_repair_block_l55',
            'upstream_repair_random_block_l55',
        )
    ]
    norm_audit_rows.append({
        'scene_id': keys[0], 'mapping_variant': keys[1],
        'semantic_relative_difference': abs(
            semantic_norms[0] - semantic_norms[1]
        ) / semantic_norms[0],
        'code_relative_difference': abs(
            code_norms[0] - code_norms[1]
        ) / code_norms[0],
    })
norm_audit_frame = pd.DataFrame(norm_audit_rows)
assert float(norm_audit_frame[
    ['semantic_relative_difference', 'code_relative_difference']
].to_numpy().max()) < NORM_RELATIVE_TOLERANCE
norm_audit_frame.to_csv(ANALYSIS_DIR / 'axis_norm_audit.csv', index=False)

audit_summary = {
    'matched_quadruplets': MATCHED_UNITS,
    'scenes': len(scene_manifest),
    'records': len(record_frame),
    'exact_physical_matching': True,
    'maximum_absolute_physical_smd': 0.0,
    'current_interventions_used_for_selection': False,
    'physical_configuration_overlap_with_parent': 0,
    'render_style_overlap_with_parent': 0,
    'replication_power_gate': REPLICATION_POWER_GATE,
    'layers_directions_doses_refit_on_replication': False,
    'scene_cohort_is_new': True,
    'intervention_family_is_new': False,
    'baseline_replay_concordance': BASELINE_REPLAY_CONCORDANCE,
    'minimum_cell_replay_concordance': MINIMUM_CELL_REPLAY_CONCORDANCE,
    'baseline_replay_gate': BASELINE_REPLAY_GATE,
    'maximum_self_replay_centered_error': max(self_errors.values()),
    'maximum_axis_norm_relative_difference': float(norm_audit_frame[
        ['semantic_relative_difference', 'code_relative_difference']
    ].to_numpy().max()),
    'claim_boundary': (
        'This is an independent physical-configuration and render replication of a '
        'previously frozen intervention. A positive result supports transport through '
        'the tested one-dimensional coordinates, not an exhaustive unique circuit.'
    ),
}
write_json_atomic(ANALYSIS_DIR / 'independent_replication_audit.json', audit_summary)

decision = {
    'primary_decision': PRIMARY_DECISION,
    'preliminary_decision': PRELIMINARY_DECISION,
    'primary_gates': primary_gates,
    'robustness_gates': robustness_gates,
    'primary_results': {
        name: result(name) for name in (
            'upstream_total', 'upstream_l49_block_loss',
            'upstream_l49_specific_loss', 'upstream_l55_block_loss',
            'upstream_l55_specific_loss', 'l49_l55_block_loss',
            'l49_l55_specific_loss', 'l55_direct', 'l55_bypass_gain',
        )
    },
    'audit': audit_summary,
    'interpretation_rule': {
        'ordered_positive': (
            'specific losses at L49 and L55 plus L49-to-L55 specific loss '
            'and successful late bypass'
        ),
        'late_only': 'L55-specific losses without a reliable L49 edge',
        'null': 'physical-axis blocking does not exceed random-axis blocking',
    },
}
write_json_atomic(ANALYSIS_DIR / 'independent_replication_decision.json', decision)
display(audit_summary)


cell_stage('Rendering the independent replication figure and result card')
import gc
import matplotlib.pyplot as plt

def test_row(metric, contrast='right_error_mean'):
    return test_frame[
        test_frame['metric'].eq(metric)
        & test_frame['contrast'].eq(contrast)
    ].iloc[0]

def errorbar_bar(axis, labels, rows, colors, title, ylabel):
    estimates = np.asarray([row['estimate'] for row in rows], dtype=float)
    lower = estimates - np.asarray([row['ci_low'] for row in rows], dtype=float)
    upper = np.asarray([row['ci_high'] for row in rows], dtype=float) - estimates
    axis.bar(labels, estimates, color=colors,
             yerr=np.vstack([lower, upper]), capsize=4)
    axis.axhline(0, color='black', linewidth=0.8)
    axis.set_title(title); axis.set_ylabel(ylabel)
    axis.tick_params(axis='x', rotation=20)

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
baseline_accuracy = baseline_frame.groupby(['cohort', 'side'])['correct'].mean()
x = np.arange(len(COHORTS)); width = 0.34
for index, side in enumerate(SIDES):
    axes[0, 0].bar(
        x + (index - 0.5) * width,
        [baseline_accuracy.loc[(cohort, side)] for cohort in COHORTS],
        width, label=side.title(),
        color=('#2563EB' if side == 'left' else '#D97706'),
    )
axes[0, 0].set_xticks(x, ['Both correct', 'Right error'])
axes[0, 0].set_ylim(0, 1.05)
axes[0, 0].set_title('A. Frozen behavior cells')
axes[0, 0].set_ylabel('Baseline accuracy')
axes[0, 0].legend(frameon=False)

errorbar_bar(
    axes[0, 1], ['L49 block', 'L49 random'],
    [test_row('upstream_l49_block_loss'),
     test_row('upstream_l49_random_loss')],
    ['#7C3AED', '#94A3B8'], 'B. L48 effect carried through L49',
    'Lost correct-answer log-odds',
)
errorbar_bar(
    axes[0, 2], ['L55 block', 'L55 random'],
    [test_row('upstream_l55_block_loss'),
     test_row('upstream_l55_random_loss')],
    ['#DC2626', '#94A3B8'], 'C. L48 effect carried through L55',
    'Lost correct-answer log-odds',
)
errorbar_bar(
    axes[1, 0], ['L55 block', 'L55 random'],
    [test_row('l49_l55_block_loss'),
     test_row('l49_l55_random_loss')],
    ['#DC2626', '#94A3B8'], 'D. L49 effect carried through L55',
    'Lost correct-answer log-odds',
)

right_error_rescue = rescue_frame[
    rescue_frame['cohort'].eq('right_error')
    & rescue_frame['side'].eq('right')
].set_index('condition')
rescue_conditions = [
    'upstream_repair', 'upstream_repair_block_l49',
    'upstream_repair_block_l55', 'l55_repair',
    'l55_bypass_after_l49_block',
]
axes[1, 1].bar(
    ['L48 repair', '+L49 block', '+L55 block',
     'L55 repair', 'L55 bypass'],
    [right_error_rescue.loc[name, 'estimate'] for name in rescue_conditions],
    color=['#2563EB', '#7C3AED', '#DC2626', '#0F766E', '#D97706'],
)
axes[1, 1].set_ylim(0, 1.05)
axes[1, 1].set_ylabel('Right-error rescue rate')
axes[1, 1].set_title('E. Block and downstream bypass')
axes[1, 1].tick_params(axis='x', rotation=25)

mapping_plot = mapping_frame[
    mapping_frame['metric'].eq('upstream_l55_specific_loss')
].set_index('mapping_variant').loc[list(MAPPING_VARIANTS)]
errorbar_bar(
    axes[1, 2], ['Forward map', 'Reverse map'],
    [mapping_plot.loc[variant] for variant in MAPPING_VARIANTS],
    ['#0F766E', '#D97706'], 'F. Answer-code robustness',
    'L55 physical-minus-random loss',
)

fig.suptitle(PRIMARY_DECISION, y=1.01)
fig.tight_layout()
figure_path = FIGURE_DIR / 'sequential_handoff_independent_replication.png'
fig.savefig(figure_path, dpi=220, bbox_inches='tight')
plt.show()

def interval(metric, contrast='right_error_mean'):
    row = test_row(metric, contrast)
    return f"{row['estimate']:.5f} [{row['ci_low']:.5f}, {row['ci_high']:.5f}]"

def rescue_interval(condition):
    row = right_error_rescue.loc[condition]
    return f"{row['estimate']:.3f} [{row['ci_low']:.3f}, {row['ci_high']:.3f}]"

lines_out = [
    '# Qwen-27B Sequential Handoff Independent Replication', '',
    f'**Primary decision:** `{PRIMARY_DECISION}`', '',
    '## Design', '',
    f'- Independent exact-physics matched quadruplets: {MATCHED_UNITS}',
    '- Frozen intervention family: repair an upstream coordinate, then restore one downstream coordinate to its untreated value.',
    '- Every mediated effect subtracts the corresponding clamp-only effect before comparing physical and random axes.',
    '- Physical-axis clamps are compared with deterministic norm-matched random-axis clamps.',
    '- No scene is selected using hidden states or the present intervention outcomes.',
    '- Every parent physical geometry and render style is excluded before screening.', '',
    '## Ordered path tests in right-side errors', '',
    f"- L48 repair total effect: {interval('upstream_total')}",
    f"- L48 effect removed by L49 clamp: {interval('upstream_l49_block_loss')}",
    f"- L49 physical clamp minus random clamp: {interval('upstream_l49_specific_loss')}",
    f"- L48 effect removed by L55 clamp: {interval('upstream_l55_block_loss')}",
    f"- L55 physical clamp minus random clamp: {interval('upstream_l55_specific_loss')}",
    f"- L49 effect removed by L55 clamp: {interval('l49_l55_block_loss')}",
    f"- L49-to-L55 physical minus random loss: {interval('l49_l55_specific_loss')}",
    f"- Direct L55 repair: {interval('l55_direct')}",
    f"- L55 bypass after L49 block: {interval('l55_bypass_gain')}", '',
    '## Behavioral rescue', '',
    f"- L48 repair: {rescue_interval('upstream_repair')}",
    f"- L48 repair plus L49 block: {rescue_interval('upstream_repair_block_l49')}",
    f"- L48 repair plus L55 block: {rescue_interval('upstream_repair_block_l55')}",
    f"- Direct L55 repair: {rescue_interval('l55_repair')}",
    f"- L55 bypass after L49 block: {rescue_interval('l55_bypass_after_l49_block')}", '',
    '## Claim boundary', '',
    (
        'A positive result independently replicates ordered transport through the '
        'frozen L49 physical coordinate and L55 response coordinate on new physics '
        'and renderings. It does not establish that the full model uses only one '
        'serial circuit.'
    ), '',
]
report_path = ANALYSIS_DIR / 'RESULT_CARD.md'
report_path.write_text('\n'.join(lines_out), encoding='utf-8')

run_complete = {
    'status': 'complete',
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'primary_decision': PRIMARY_DECISION,
    'matched_quadruplets': MATCHED_UNITS,
    'scene_cohort_is_new': True,
    'intervention_family_is_new': False,
    'exact_physical_matching': True,
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'scene_manifest_sha256': sha256_file(SCENE_MANIFEST_PATH),
    'decision_sha256': sha256_file(
        ANALYSIS_DIR / 'independent_replication_decision.json'
    ),
    'figure': str(figure_path), 'report': str(report_path),
}
write_json_atomic(OUTPUT_ROOT / 'RUN_COMPLETE.json', run_complete)

if 'adapter' in globals():
    del adapter
gc.collect()
torch.cuda.empty_cache()
print('Saved:', figure_path)
print('Saved:', report_path)
print('RUN_COMPLETE:', OUTPUT_ROOT / 'RUN_COMPLETE.json')
